# Full Planner — Best Strategy Simulation Notebook

This notebook is dedicated to simulating, evaluating, and discovering the **best overall strategy** for SpaceInvasion Classic across full-empire progression (economy scaling, energy optimization, research unlocks, fleet composition, raiding, colonization, and action scheduling).

## 0. Package Installation

Install required third-party Python dependencies (`numpy`, `pandas`, `beautifulsoup4`, `python-dotenv`, `requests`) in the active Jupyter or Colab kernel before running the notebook.

In [ ]:
!pip install -q "numpy>=1.26,<3" "pandas>=2.2,<4" "beautifulsoup4>=4.12,<5" "python-dotenv>=1.0,<2" "requests>=2.31,<3"

## 0.1 Imports

All Python standard-library, `numpy` / `pandas`, and `spaceinvasion` domain imports used across the simulation notebook.

In [ ]:
from collections import deque
import contextlib
import dataclasses
import datetime
import json
import math
from pathlib import Path
import shutil
import sqlite3
import sys
import tempfile
from typing import Any

import numpy as np
import pandas as pd

for _cand in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_cand / "spaceinvasion" / "__init__.py").is_file():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break

from spaceinvasion import game_data
from spaceinvasion.action_graph import build_action_graph, merge_strategy_config_overrides
from spaceinvasion.config import StrategyConfig
from spaceinvasion.fleet import (
    DEFENSE_SLUG_BY_KEY,
    SHIP_SLUG_BY_KEY,
    _parse_coords_distance,
)
from spaceinvasion.logging_utils import redact_sensitive
from spaceinvasion.optimizer import encode_empire_state, run_optimizer, simulate_action_sequence
from spaceinvasion.ranking import (
    POINTS_PER_RESOURCE_DIVISOR,
    attach_action_etas,
    compute_action_eta,
    estimate_account_points,
    estimate_hourly_economy,
)
from spaceinvasion.state import EnergyBalance, GameState, PlanetState, Resources
from spaceinvasion.storage import GameDatabase
from spaceinvasion.strategy import PlannedAction, plan_actions

## 1. Utils

Shared path-resolution (`resolve_repo_root`, `resolve_active_data_root`), formatting (`fmt_duration`, `render_table`), JSON parsing (`_safe_json_dict`), statistical (`compute_mean_and_std`), flight-physics (`estimate_physics_one_way_seconds`), and full-data printing (`format_full_float`, `render_data_item`, `print_section_data`) utilities reused across all notebook sections.

In [ ]:
pd.set_option("display.max_columns", 28)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 64)


def resolve_repo_root() -> Path:
    """Locate repository root from any working directory without bare `__file__`."""
    cwd = Path.cwd().resolve()
    for cand in (cwd, *cwd.parents):
        if (cand / "spaceinvasion" / "__init__.py").is_file():
            return cand
    for entry in list(sys.path):
        if not entry:
            continue
        p = Path(entry).resolve()
        if (p / "spaceinvasion" / "__init__.py").is_file():
            return p
    module_path = Path(game_data.__file__).resolve()
    for cand in module_path.parents:
        if (cand / "spaceinvasion" / "__init__.py").is_file():
            return cand
    return module_path.parents[1]


def resolve_active_data_root(repo_root: Path) -> Path:
    """Prefer isolated `cwd` data/state fixtures when present; otherwise use `repo_root`."""
    cwd = Path.cwd().resolve()
    if (cwd / "state" / "spaceinvasion.db").is_file() or (
        cwd / "data" / "attack_history.json"
    ).is_file():
        return cwd
    return repo_root


def fmt_duration(seconds: float) -> str:
    """Format seconds into readable `Xd Yh`, `Xh Ym`, or `Xm Ys` strings."""
    if not math.isfinite(seconds) or seconds <= 0.0:
        return "0s"
    sec_i = int(round(seconds))
    if sec_i < 60:
        return f"{sec_i}s"
    mins, rem_s = divmod(sec_i, 60)
    if mins < 60:
        return f"{mins}m {rem_s:02d}s"
    hrs, rem_m = divmod(mins, 60)
    if hrs < 48:
        return f"{hrs}h {rem_m:02d}m"
    days, rem_h = divmod(hrs, 24)
    return f"{days}d {rem_h:02d}h"


def render_table(headers: list[str], rows: list[list[Any]]) -> str:
    """Render an aligned ASCII grid."""
    str_rows = [[str(c) for c in row] for row in rows]
    widths = [len(h) for h in headers]
    for row in str_rows:
        for idx, cell in enumerate(row):
            if idx < len(widths):
                widths[idx] = max(widths[idx], len(cell))
    sep = "+-" + "-+-".join("-" * w for w in widths) + "-+"
    head = "| " + " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers)) + " |"
    body = [
        "| " + " | ".join((row[i] if i < len(row) else "").ljust(widths[i]) for i in range(len(widths))) + " |"
        for row in str_rows
    ]
    return "\n".join([sep, head, sep, *body, sep])


def _safe_json_dict(raw: Any) -> dict[str, Any]:
    if isinstance(raw, dict):
        return dict(raw)
    if isinstance(raw, str) and raw.strip():
        try:
            parsed = json.loads(raw)
            return dict(parsed) if isinstance(parsed, dict) else {}
        except ValueError:
            return {}
    return {}


def compute_mean_and_std(values: Any) -> tuple[float, float, float]:
    """Return `(mean, sample_std_ddof1, pop_std_ddof0)` via vectorized `numpy`."""
    arr = np.asarray(values, dtype=np.float64).ravel()
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return 0.0, 0.0, 0.0
    mean_val = float(np.mean(arr))
    sample_std = float(np.std(arr, ddof=1)) if arr.size >= 2 else 0.0
    pop_std = float(np.std(arr, ddof=0))
    return mean_val, sample_std, pop_std


def estimate_physics_one_way_seconds(
    origin_coords: str,
    target_coords: str,
    *,
    min_ship_speed: float = 12_000.0,
    fleet_speed_factor: float = 5.0,
) -> float:
    """SpaceInvasion Classic one-way flight time formula in seconds."""
    dist = max(1.0, float(_parse_coords_distance(origin_coords, target_coords)))
    spd = max(100.0, float(min_ship_speed))
    fac = max(100.0 if fleet_speed_factor > 100.0 else max(1.0, float(fleet_speed_factor)), 1.0)
    return float(round(10.0 + (3500.0 / fac) * math.sqrt((10.0 * dist) / spd)))


def format_prerequisites(requirements: list[dict[str, Any]] | tuple[dict[str, Any], ...]) -> str:
    """Format tech-tree requirement dicts into compact `b7 L3, r2 L1` notation."""
    if not requirements:
        return "None"
    parts: list[str] = []
    for req in requirements:
        cat = str(req.get("category") or "")
        slug = str(req.get("slug") or "")
        lvl = int(req.get("level") or 0)
        if cat == "buildings":
            b_info = game_data.BUILDINGS.get(slug) or {}
            bid = b_info.get("id")
            code = f"b{bid}" if bid is not None else slug
        elif cat == "research":
            r_info = game_data.RESEARCHES.get(slug) or {}
            rid = r_info.get("id")
            code = str(rid) if rid else slug
        else:
            code = slug or str(req.get("name") or "?")
        parts.append(f"{code} L{lvl} ({req.get('name', slug)})")
    return ", ".join(parts)


def build_monotonic_base_times(levels_map: dict[int, dict[str, Any]], max_level: int) -> dict[int, float]:
    """Extract level-by-level `base_time_s`, repairing high-level wiki scientific-notation mantissa drops."""
    out: dict[int, float] = {}
    prev_time = 0.0
    prev_cost_sp = 0.0
    for lvl in range(1, max_level + 1):
        row = levels_map.get(lvl) or {}
        raw_t = float(row.get("base_time_s") or 0.0)
        cost_sp = float(game_data.spice_equivalent(row.get("cost") or {}))
        if lvl > 1 and prev_time > 0.0 and raw_t < prev_time:
            ratio = (cost_sp / prev_cost_sp) if prev_cost_sp > 0.0 else 1.51
            fixed_t = prev_time * max(1.05, ratio)
        else:
            fixed_t = max(0.0, raw_t)
        out[lvl] = fixed_t
        prev_time = fixed_t
        prev_cost_sp = cost_sp
    return out


def solar_cell_energy_per_unit(temperature_c: float) -> float:
    """Orbital Solar cell (`s9`) base energy output (`-160..-80C: 10`, `0C: 30`, `+40C: 40`, `+200C: 80`)."""
    clamped_t = max(-80.0, min(200.0, float(temperature_c)))
    return float(max(10.0, min(80.0, 30.0 + 0.25 * clamped_t)))


def format_full_float(value: float) -> str:
    """Format a float for full printouts without scientific notation for large values.

    `|x| >= 1` keeps its shortest exact decimal text, capped at 6 decimals; `|x| < 1` keeps 6 significant digits.
    """
    val = float(value)
    if not math.isfinite(val):
        return str(val)
    if val == 0.0:
        return "0"
    magnitude = abs(val)
    if magnitude < 1.0:
        return f"{val:.6g}"
    if magnitude >= 1e16:
        return f"{val:.0f}"
    whole, _, frac = repr(val).partition(".")
    if len(frac) > 6:
        return f"{val:.6f}".rstrip("0").rstrip(".")
    return whole if frac in ("", "0") else f"{whole}.{frac}"


def _float_column_formatter(column: pd.Series) -> Any:
    """Return one formatter for a float column so that its decimal points line up."""
    texts = [format_full_float(v) for v in column.to_numpy(dtype=np.float64) if math.isfinite(v)]
    if any("e" in text for text in texts):
        return format_full_float
    decimals = max((len(text.partition(".")[2]) for text in texts), default=0)
    return lambda v: f"{float(v) + 0.0:.{decimals}f}"


def _json_default(obj: Any) -> Any:
    """Convert numpy values, dataclasses, and sets for `json.dumps`; fall back to `str`."""
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if dataclasses.is_dataclass(obj) and not isinstance(obj, type):
        return dataclasses.asdict(obj)
    if isinstance(obj, (set, frozenset)):
        return sorted(obj, key=str)
    return str(obj)


def _compact_json(value: Any) -> str:
    """Single-line JSON text for one value (falls back to `repr` for unserializable objects)."""
    try:
        return json.dumps(value, default=_json_default)
    except (TypeError, ValueError):
        return repr(value)


def describe_data_object(obj: Any) -> str:
    """One-line type and size summary of a generated object."""
    if isinstance(obj, pd.DataFrame):
        return f"pandas.DataFrame, shape=({obj.shape[0]}, {obj.shape[1]})"
    if isinstance(obj, np.ndarray):
        return f"numpy.ndarray, shape={obj.shape}, dtype={obj.dtype}"
    if isinstance(obj, (dict, list, tuple, set, frozenset)):
        return f"{type(obj).__name__}, len={len(obj)}"
    return type(obj).__name__


def render_data_item(name: str, obj: Any, note: str = "") -> str:
    """Render one named object in full: every row, column, element, and cell text, with no `...` truncation.

    A non-empty `note` marks a plain-Python mirror of another printed object; it is summarized on one line.
    """
    head = f"[{name}] {describe_data_object(obj)}"
    if note:
        return f"{head} - {note}"
    if isinstance(obj, pd.DataFrame):
        if obj.empty:
            return f"{head}\n(no rows) columns: {list(obj.columns)}"
        formatters = (
            {
                col: _float_column_formatter(obj[col])
                for col in obj.columns
                if pd.api.types.is_float_dtype(obj[col])
            }
            if obj.columns.is_unique
            else None
        )
        table = obj.to_string(
            index=not isinstance(obj.index, pd.RangeIndex),
            formatters=formatters,
            float_format=format_full_float,
        )
        return f"{head}\n{table}"
    if isinstance(obj, np.ndarray):
        body = np.array2string(
            obj,
            threshold=obj.size + 1,
            max_line_width=160,
            formatter={"float_kind": format_full_float},
        )
        return f"{head}\n{body}"
    if isinstance(obj, dict):
        if not obj:
            return f"{head}\n{{}}"
        if all(isinstance(v, pd.DataFrame) for v in obj.values()):
            rows = [f"  {key}: shape=({tbl.shape[0]}, {tbl.shape[1]})" for key, tbl in obj.items()]
            return f"{head} (registry of DataFrames; each table is printed in full by its own section)\n" + "\n".join(rows)
        rows = [f"  {json.dumps(str(key))}: {_compact_json(value)}" for key, value in obj.items()]
        return f"{head}\n{{\n" + ",\n".join(rows) + "\n}"
    if isinstance(obj, (list, tuple)):
        if all(v is None or isinstance(v, (str, int, float, bool)) for v in obj):
            return f"{head}\n{_compact_json(list(obj))}"
        return f"{head} (one JSON value per line)\n" + "\n".join(_compact_json(v) for v in obj)
    return f"{head} = {obj!r}"


def print_section_data(section_title: str, items: list[tuple[Any, ...]]) -> None:
    """Print everything one section generated, in full, stamped with the date and time of the printout.

    Each item is `(name, obj)` or `(name, obj, note)` (see `render_data_item`). The whole text passes
    through `redact_sensitive` before printing so session tokens can never reach a notebook output.
    """
    printed_at = datetime.datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %z")
    rule = "=" * 100
    header = f"{rule}\n{section_title}\nPrinted: {printed_at} | {len(items)} item(s)\n{rule}"
    blocks = [
        render_data_item(str(item[0]), item[1], str(item[2]) if len(item) > 2 else "")
        for item in items
    ]
    print(redact_sensitive(header + "\n" + ("\n" + "-" * 100 + "\n").join(blocks)))


REPO_ROOT = resolve_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
ACTIVE_DATA_ROOT = resolve_active_data_root(REPO_ROOT)
LIVE_DB_PATH = ACTIVE_DATA_ROOT / "state" / "spaceinvasion.db"
ATTACK_HISTORY_PATH = ACTIVE_DATA_ROOT / "data" / "attack_history.json"
RESOURCE_KEYS: tuple[str, ...] = ("pig_iron", "metal", "kryptonite", "spice")

print(f"Utils initialized (REPO_ROOT={REPO_ROOT}, ACTIVE_DATA_ROOT={ACTIVE_DATA_ROOT})")

## 2. MER-Aligned Read-Only SQLite Telemetry Queries

Following [docs/database-mer.md](../docs/database-mer.md), this cell copies `state/spaceinvasion.db` (+ `-wal` / `-shm` sidecars) into a temporary directory and executes **four relational SQL queries** leveraging the exact keys, soft-flags, and secondary indexes of the 22-table schema:

1. **Colony & Fleet-Slot Capacity (`PLANET_SNAPSHOTS` via `idx_planet_snapshots_pid_ts`)**: Extracts latest colony coordinates, `Fleet base` (`b10`) level (`1 + b10` concurrent flight slots), and engine research levels (`r4`, `r5`, `r6`).
2. **Resolved Combat & Target Classification Join (`BATTLE_REPORTS` $\bowtie$ `GAME_MESSAGES` on `msg_uid` $\bowtie$ `SPY_TARGETS` on `coords`)**: Joins 4-resource plunder, combat fleet losses (`attacker_lost_spice`), and spawned orbital rubble (`debris_spice_eq`) with target status (`is_spacia`, `is_inactive`, `status_color`, `bash_hits`), filtering out soft-merged/own rows (`is_own_planet = 0 AND merged_into = ''`).
3. **Dispatch-to-Origin Colony Join (`RAID_HISTORY` via `idx_raid_history_coords_ts` $\bowtie$ `ACTION_HISTORY` via `idx_action_history_ts`)**: Resolves the exact launching colony `origin_coords` for each dispatched raid rather than assuming a single static origin.
4. **Orbital Rubble & Bash Cooldown Context (`RECYCLE_REPORTS` via `idx_recycle_reports_coords_ts` & `BASH_BLOCKS` via `idx_bash_blocks_coords_ts`)**: Queries supplementary recycler harvests and active server bash blocks.

In [ ]:
SQL_LATEST_COLONY_SNAPSHOT = """
SELECT coords, temperature, fields_used, fields_max, buildings_json, researches_json
FROM planet_snapshots
ORDER BY ts DESC, id DESC
LIMIT 1
"""

SQL_LATEST_GAME_STATE_META = """
SELECT value
FROM meta
WHERE key = 'latest_game_state'
LIMIT 1
"""

SQL_BATTLE_REPORTS_MER_JOIN = """
SELECT
    br.id,
    br.ts,
    br.msg_uid,
    br.battle_id,
    br.msg_date_text,
    br.target_coords,
    br.outcome,
    br.rounds,
    br.pig_iron,
    br.metal,
    br.kryptonite,
    br.spice,
    br.loot_spice_eq,
    br.attacker_lost_spice,
    br.defender_lost_spice,
    br.debris_spice_eq,
    gm.ingested_ts AS msg_ingested_ts,
    gm.deleted_in_game,
    COALESCE(st.status_color, 'unknown') AS target_status_color,
    COALESCE(st.is_inactive, 0) AS target_is_inactive,
    COALESCE(st.is_spacia, 0) AS target_is_spacia,
    COALESCE(st.bash_hits, 0) AS target_bash_hits
FROM battle_reports br
LEFT JOIN game_messages gm ON gm.msg_uid = br.msg_uid
LEFT JOIN spy_targets st
       ON st.coords = br.target_coords
      AND st.is_own_planet = 0
      AND st.merged_into = ''
ORDER BY br.msg_date_text ASC, br.id ASC
"""

SQL_RAID_DISPATCH_ORIGIN_JOIN = """
SELECT
    rh.id,
    rh.target_coords,
    rh.attacked_ts,
    rh.is_exempt,
    rh.outcome,
    rh.loot_json,
    COALESCE(
        (
            SELECT ah.planet_coords
            FROM action_history ah
            WHERE ah.kind = 'ATTACK'
              AND ah.executed = 1
              AND ah.dry_run = 0
              AND ah.planet_coords IS NOT NULL
              AND ah.planet_coords != ''
              AND ABS(ah.ts - rh.attacked_ts) <= 5.0
            ORDER BY ABS(ah.ts - rh.attacked_ts) ASC
            LIMIT 1
        ),
        ?
    ) AS origin_coords
FROM raid_history rh
ORDER BY rh.attacked_ts ASC, rh.id ASC
"""

SQL_RECYCLE_HARVESTS = """
SELECT
    coords,
    recycler_count,
    harvested_pig_iron,
    harvested_metal,
    harvested_kryptonite,
    harvested_spice,
    harvested_spice_eq,
    ts
FROM recycle_reports
ORDER BY ts ASC, id ASC
"""

SQL_ACTIVE_BASH_BLOCKS = """
SELECT COUNT(DISTINCT target_coords)
FROM bash_blocks
WHERE blocked_until_ts > ?
"""

ATTACK_DF_COLUMNS: tuple[str, ...] = (
    "event_ts",
    "origin_coords",
    "target_coords",
    "target_class",
    "won",
    "outcome",
    "pig_iron",
    "metal",
    "kryptonite",
    "spice",
    "spice_eq",
    "attacker_lost_spice",
    "net_spice_eq",
    "debris_spice_eq",
    "one_way_sec",
    "round_trip_sec",
    "time_source",
)

RECYCLE_DF_COLUMNS: tuple[str, ...] = (
    "coords",
    "recycler_count",
    "harvested_pig_iron",
    "harvested_metal",
    "harvested_kryptonite",
    "harvested_spice",
    "harvested_spice_eq",
    "ts",
)


def load_attack_telemetry_readonly(
    db_path: Path,
    json_path: Path,
) -> dict[str, Any]:
    """Execute static MER SQL queries on an isolated temp copy of `spaceinvasion.db`."""
    origin_coords = "02:380:12"
    colony_temperature = 31
    colony_buildings: dict[str, int] = {}
    colony_researches: dict[str, int] = {}
    universe_speedup_factor = 5.0
    fleet_base_level = 0
    engine_tech_levels: dict[str, int] = {"r4": 0, "r5": 0, "r6": 0}
    battle_rows: list[dict[str, Any]] = []
    raid_rows: list[dict[str, Any]] = []
    recycle_rows: list[dict[str, Any]] = []
    active_bash_blocks: int = 0
    source_label = "cold_start"

    if db_path.is_file() and db_path.stat().st_size >= 100:
        with tempfile.TemporaryDirectory(prefix="si_fp_ro_") as tmp_dir:
            tmp_root = Path(tmp_dir)
            tmp_db = tmp_root / db_path.name
            shutil.copy2(db_path, tmp_db)
            tmp_db.chmod(0o600)
            for suffix in ("-wal", "-shm", "-journal"):
                sidecar = Path(f"{db_path}{suffix}")
                if sidecar.is_file():
                    tmp_sidecar = Path(f"{tmp_db}{suffix}")
                    shutil.copy2(sidecar, tmp_sidecar)
                    tmp_sidecar.chmod(0o600)

            ro_db = GameDatabase(
                tmp_db,
                attack_history_path=tmp_root / "ro_mirror.json",
                backup_on_upgrade=False,
            )
            try:
                conn = ro_db._conn
                ps = conn.execute(SQL_LATEST_COLONY_SNAPSHOT).fetchone()
                if ps:
                    if ps["coords"]:
                        origin_coords = str(ps["coords"])
                    if ps["temperature"] is not None:
                        colony_temperature = int(ps["temperature"])
                    bld_d = _safe_json_dict(ps["buildings_json"])
                    res_d = _safe_json_dict(ps["researches_json"])
                    colony_buildings = {
                        str(k): max(0, int(v or 0))
                        for k, v in bld_d.items()
                        if isinstance(v, (int, float))
                    }
                    colony_researches = {
                        str(k): max(0, int(v or 0))
                        for k, v in res_d.items()
                        if isinstance(v, (int, float))
                    }
                    fleet_base_level = max(0, int(colony_buildings.get("b10") or 0))
                    for rk in ("r4", "r5", "r6"):
                        engine_tech_levels[rk] = max(0, int(colony_researches.get(rk) or 0))

                meta_row = conn.execute(SQL_LATEST_GAME_STATE_META).fetchone()
                if meta_row and meta_row[0]:
                    gs_dict = _safe_json_dict(meta_row[0])
                    raw_spd = gs_dict.get("speedup_factor")
                    if isinstance(raw_spd, (int, float)) and math.isfinite(float(raw_spd)) and float(raw_spd) > 0.0:
                        universe_speedup_factor = float(raw_spd)

                battle_rows = [dict(r) for r in conn.execute(SQL_BATTLE_REPORTS_MER_JOIN).fetchall()]
                raid_rows = [
                    dict(r)
                    for r in conn.execute(SQL_RAID_DISPATCH_ORIGIN_JOIN, (origin_coords,)).fetchall()
                ]
                recycle_rows = [dict(r) for r in conn.execute(SQL_RECYCLE_HARVESTS).fetchall()]

                ref_ts = max(
                    [0.0]
                    + [float(r.get("attacked_ts") or 0.0) for r in raid_rows]
                    + [float(b.get("ts") or 0.0) for b in battle_rows]
                )
                active_bash_blocks = int(
                    conn.execute(SQL_ACTIVE_BASH_BLOCKS, (ref_ts,)).fetchone()[0]
                )
            finally:
                ro_db.close()

            if battle_rows or raid_rows:
                source_label = "sqlite:mer_joined(battle_reports+raid_history+spy_targets)"

    if not battle_rows and not raid_rows and json_path.is_file():
        raw_obj = _safe_json_dict(json_path.read_text(encoding="utf-8"))
        for row in raw_obj.get("history") or []:
            if isinstance(row, dict):
                raid_rows.append({**dict(row), "origin_coords": origin_coords})
        if raid_rows:
            source_label = "json:attack_history"

    dispatch_timestamps = sorted(
        float(r["attacked_ts"])
        for r in raid_rows
        if isinstance(r.get("attacked_ts"), (int, float)) and math.isfinite(float(r["attacked_ts"]))
    )

    tz_offsets_sec: list[float] = []
    parsed_battle_naive: list[float | None] = []
    for brow in battle_rows:
        dt_txt = str(brow.get("msg_date_text") or "").strip()
        try:
            naive_ts = (
                datetime.datetime.strptime(dt_txt, "%Y-%m-%d %H:%M:%S")
                .replace(tzinfo=datetime.timezone.utc)
                .timestamp()
            )
        except ValueError:
            naive_ts = None
        parsed_battle_naive.append(naive_ts)
        if naive_ts is not None:
            for rrow in raid_rows:
                if str(rrow.get("target_coords") or "") != str(brow.get("target_coords") or ""):
                    continue
                rts = float(rrow.get("attacked_ts") or 0.0)
                cand_h = round((rts - naive_ts) / 3600.0)
                if abs(cand_h) <= 14 and 30.0 <= (naive_ts + cand_h * 3600.0 - rts) <= 3600.0:
                    tz_offsets_sec.append(cand_h * 3600.0)

    server_tz_offset_sec = (
        float(np.median(np.asarray(tz_offsets_sec, dtype=np.float64)))
        if tz_offsets_sec
        else 0.0
    )

    records: list[dict[str, Any]] = []
    used_raid_ids: set[int] = set()
    if battle_rows:
        matched_pairs_sec_per_sqrtdist: list[float] = []
        provisional_rows: list[tuple[dict[str, Any], float, float | None, str]] = []
        for brow, naive_ts in zip(battle_rows, parsed_battle_naive):
            t_coords = str(brow.get("target_coords") or "")
            impact_ts = (
                (naive_ts + server_tz_offset_sec)
                if naive_ts is not None
                else float(brow.get("ts") or 0.0)
            )
            matched_one_way: float | None = None
            matched_origin = origin_coords
            best_rid: int | None = None
            for idx_r, rrow in enumerate(raid_rows):
                if idx_r in used_raid_ids or str(rrow.get("target_coords") or "") != t_coords:
                    continue
                rts = float(rrow.get("attacked_ts") or 0.0)
                dt_flight = impact_ts - rts
                if 30.0 <= dt_flight <= 7200.0:
                    if matched_one_way is None or dt_flight < matched_one_way:
                        matched_one_way = dt_flight
                        matched_origin = str(rrow.get("origin_coords") or origin_coords)
                        best_rid = idx_r
            if best_rid is not None and matched_one_way is not None:
                used_raid_ids.add(best_rid)
                dist = max(1.0, float(_parse_coords_distance(matched_origin, t_coords)))
                matched_pairs_sec_per_sqrtdist.append(max(1.0, matched_one_way - 10.0) / math.sqrt(dist))
            provisional_rows.append((brow, impact_ts, matched_one_way, matched_origin))

        emp_scale = (
            float(np.median(np.asarray(matched_pairs_sec_per_sqrtdist, dtype=np.float64)))
            if matched_pairs_sec_per_sqrtdist
            else None
        )
        for brow, impact_ts, matched_one_way, row_origin in provisional_rows:
            t_coords = str(brow.get("target_coords") or "")
            if matched_one_way is not None:
                one_way_s = matched_one_way
                t_src = "observed_dispatch_pair"
            elif emp_scale is not None:
                dist = max(1.0, float(_parse_coords_distance(row_origin, t_coords)))
                one_way_s = 10.0 + emp_scale * math.sqrt(dist)
                t_src = "calibrated_distance"
            else:
                one_way_s = estimate_physics_one_way_seconds(row_origin, t_coords)
                t_src = "physics_formula"

            pig = float(brow.get("pig_iron") or 0.0)
            met = float(brow.get("metal") or 0.0)
            kry = float(brow.get("kryptonite") or 0.0)
            spi = float(brow.get("spice") or 0.0)
            sp_eq = float(
                brow.get("loot_spice_eq")
                if brow.get("loot_spice_eq") is not None
                else game_data.spice_equivalent({"pig_iron": pig, "metal": met, "kryptonite": kry, "spice": spi})
            )
            lost_sp = float(brow.get("attacker_lost_spice") or 0.0)
            debris_sp = float(brow.get("debris_spice_eq") or 0.0)
            outcome_str = str(brow.get("outcome") or "").strip().lower()
            won = outcome_str in ("win", "victory")
            is_spacia = bool(brow.get("target_is_spacia"))
            is_inactive = bool(brow.get("target_is_inactive")) or str(brow.get("target_status_color") or "") == "yellow"
            target_class = "spacia" if is_spacia else ("yellow_inactive" if is_inactive else "active_colony")
            records.append(
                {
                    "event_ts": impact_ts,
                    "origin_coords": row_origin,
                    "target_coords": t_coords,
                    "target_class": target_class,
                    "won": won,
                    "outcome": outcome_str or ("win" if won else "loss"),
                    "pig_iron": pig,
                    "metal": met,
                    "kryptonite": kry,
                    "spice": spi,
                    "spice_eq": sp_eq,
                    "attacker_lost_spice": lost_sp,
                    "net_spice_eq": sp_eq - lost_sp,
                    "debris_spice_eq": debris_sp,
                    "one_way_sec": float(one_way_s),
                    "round_trip_sec": float(2.0 * one_way_s),
                    "time_source": t_src,
                }
            )
    else:
        for rrow in raid_rows:
            t_coords = str(rrow.get("target_coords") or "")
            row_origin = str(rrow.get("origin_coords") or origin_coords)
            loot_d = _safe_json_dict(rrow.get("loot") if "loot" in rrow else rrow.get("loot_json"))
            pig = float(loot_d.get("pig_iron") or 0.0)
            met = float(loot_d.get("metal") or 0.0)
            kry = float(loot_d.get("kryptonite") or 0.0)
            spi = float(loot_d.get("spice") or 0.0)
            sp_eq = float(
                game_data.spice_equivalent(
                    {"pig_iron": pig, "metal": met, "kryptonite": kry, "spice": spi}
                )
            )
            outcome_txt = redact_sensitive(str(rrow.get("outcome") or "")).lower()
            won = bool(rrow.get("won")) if "won" in rrow else ("defeat" not in outcome_txt and "loss" not in outcome_txt)
            one_way_s = estimate_physics_one_way_seconds(row_origin, t_coords)
            records.append(
                {
                    "event_ts": float(rrow.get("attacked_ts") or 0.0),
                    "origin_coords": row_origin,
                    "target_coords": t_coords,
                    "target_class": "exempt_farm" if bool(rrow.get("is_exempt")) else "active_colony",
                    "won": won,
                    "outcome": "win" if won else "loss",
                    "pig_iron": pig,
                    "metal": met,
                    "kryptonite": kry,
                    "spice": spi,
                    "spice_eq": sp_eq,
                    "attacker_lost_spice": 0.0,
                    "net_spice_eq": sp_eq,
                    "debris_spice_eq": 0.0,
                    "one_way_sec": float(one_way_s),
                    "round_trip_sec": float(2.0 * one_way_s),
                    "time_source": "physics_formula",
                }
            )

    records.sort(key=lambda r: float(r["event_ts"]))
    if not dispatch_timestamps:
        dispatch_timestamps = [float(r["event_ts"]) for r in records if float(r["event_ts"]) > 0.0]
    return {
        "records": records,
        "dispatch_timestamps": dispatch_timestamps,
        "origin_coords": origin_coords,
        "colony_temperature": colony_temperature,
        "colony_buildings": colony_buildings,
        "colony_researches": colony_researches,
        "universe_speedup_factor": universe_speedup_factor,
        "fleet_base_level": fleet_base_level,
        "fleet_slot_capacity": 1 + fleet_base_level,
        "engine_tech_levels": engine_tech_levels,
        "recycle_rows": recycle_rows,
        "active_bash_blocks": active_bash_blocks,
        "source_label": source_label,
    }


telemetry_bundle = load_attack_telemetry_readonly(LIVE_DB_PATH, ATTACK_HISTORY_PATH)
attack_records: list[dict[str, Any]] = telemetry_bundle["records"]
attack_dispatch_timestamps: list[float] = telemetry_bundle["dispatch_timestamps"]
attack_dispatch_ts_arr: np.ndarray = np.asarray(attack_dispatch_timestamps, dtype=np.float64)
origin_planet_coords: str = telemetry_bundle["origin_coords"]
fleet_slot_capacity: int = int(telemetry_bundle["fleet_slot_capacity"])
recycle_records: list[dict[str, Any]] = telemetry_bundle["recycle_rows"]
telemetry_source: str = telemetry_bundle["source_label"]

attacks_df = pd.DataFrame(attack_records, columns=list(ATTACK_DF_COLUMNS))
for _num_col in (
    "event_ts",
    "pig_iron",
    "metal",
    "kryptonite",
    "spice",
    "spice_eq",
    "attacker_lost_spice",
    "net_spice_eq",
    "debris_spice_eq",
    "one_way_sec",
    "round_trip_sec",
):
    attacks_df[_num_col] = pd.to_numeric(attacks_df[_num_col], errors="coerce").fillna(0.0).astype(np.float64)
attacks_df["won"] = attacks_df["won"].astype(bool)

recycles_df = pd.DataFrame(recycle_records, columns=list(RECYCLE_DF_COLUMNS))
for _r_col in (
    "recycler_count",
    "harvested_pig_iron",
    "harvested_metal",
    "harvested_kryptonite",
    "harvested_spice",
    "harvested_spice_eq",
    "ts",
):
    recycles_df[_r_col] = pd.to_numeric(recycles_df[_r_col], errors="coerce").fillna(0.0).astype(np.float64)

print(
    render_table(
        ["MER Telemetry Property", "Value"],
        [
            ["Telemetry Source", telemetry_source],
            ["Primary Origin Colony", f"{origin_planet_coords} ({telemetry_bundle['colony_temperature']} °C, {telemetry_bundle['universe_speedup_factor']:.1f}x speed)"],
            ["Fleet Base Level / Flight Slots", f"b10 L{telemetry_bundle['fleet_base_level']} -> {fleet_slot_capacity} concurrent slots"],
            ["Engine Tech Levels (r4 / r5 / r6)", f"{telemetry_bundle['engine_tech_levels']['r4']} / {telemetry_bundle['engine_tech_levels']['r5']} / {telemetry_bundle['engine_tech_levels']['r6']}"],
            ["attacks_df Shape (Rows x Cols)", f"{attacks_df.shape[0]} x {attacks_df.shape[1]}"],
            ["Distinct Targets / Active Bash Blocks", f"{int(attacks_df['target_coords'].nunique())} targets / {telemetry_bundle['active_bash_blocks']} blocked"],
            ["recycles_df Shape (Rows x Cols)", f"{recycles_df.shape[0]} x {recycles_df.shape[1]}"],
        ],
    )
)

## 3. Mean & Standard Deviation of Resources Achieved in Attacks (`pandas` & `numpy`)

Using vectorized **`pandas` (`attacks_df`, `recycles_df`)** and **`numpy` (`np.mean`, `np.std(..., ddof=1)`, `np.std(..., ddof=0)`)** operations over the `BATTLE_REPORTS` schema ([docs/database-mer.md](../docs/database-mer.md)), this section builds **`attack_resource_stats_df`** with sample mean ($\mu_r$), sample standard deviation ($s_r$, `ddof=1`), and population standard deviation ($\sigma_r$, `ddof=0`) across:

- **Gross Plunder**: `pig_iron`, `metal`, `kryptonite`, `spice`, and `spice_eq` (`6 : 2.5 : 3 : 1` Merchant parity).
- **Net Combat Yield (`net_spice_eq`)**: $\text{loot\_spice\_eq} - \text{attacker\_lost\_spice}$, accounting for fleet losses suffered in unsuccessful or costly engagements.
- **Orbital Rubble (`debris_spice_eq` + `recycles_df`)**: Combat-spawned debris fields and verified `Recycler` (`s11`) harvests (`harvested_spice_eq`).

In [ ]:
METRIC_KEYS: tuple[str, ...] = (*RESOURCE_KEYS, "spice_eq", "net_spice_eq", "debris_spice_eq")
_metric_cols: list[str] = list(METRIC_KEYS)

winning_attacks_df = attacks_df.loc[attacks_df["won"]].reset_index(drop=True)
_n_all = int(len(attacks_df))
_n_win = int(len(winning_attacks_df))

_all_mat = attacks_df[_metric_cols].to_numpy(dtype=np.float64)
_win_mat = winning_attacks_df[_metric_cols].to_numpy(dtype=np.float64)

_mean_all_vec = np.mean(_all_mat, axis=0) if _n_all >= 1 else np.zeros(len(_metric_cols), dtype=np.float64)
_sstd_all_vec = np.std(_all_mat, axis=0, ddof=1) if _n_all >= 2 else np.zeros(len(_metric_cols), dtype=np.float64)
_pstd_all_vec = np.std(_all_mat, axis=0, ddof=0) if _n_all >= 1 else np.zeros(len(_metric_cols), dtype=np.float64)

_mean_win_vec = np.mean(_win_mat, axis=0) if _n_win >= 1 else np.zeros(len(_metric_cols), dtype=np.float64)
_sstd_win_vec = np.std(_win_mat, axis=0, ddof=1) if _n_win >= 2 else np.zeros(len(_metric_cols), dtype=np.float64)
_pstd_win_vec = np.std(_win_mat, axis=0, ddof=0) if _n_win >= 1 else np.zeros(len(_metric_cols), dtype=np.float64)

attack_resource_stats_df = pd.DataFrame(
    {
        "metric": _metric_cols,
        "samples_all": _n_all,
        "mean_all": _mean_all_vec,
        "sample_std_all": _sstd_all_vec,
        "pop_std_all": _pstd_all_vec,
        "samples_wins": _n_win,
        "mean_wins": _mean_win_vec,
        "sample_std_wins": _sstd_win_vec,
        "pop_std_wins": _pstd_win_vec,
    }
)

attack_resource_mean: dict[str, float] = {
    str(row["metric"]): float(row["mean_all"]) for _, row in attack_resource_stats_df.iterrows()
}
attack_resource_std: dict[str, float] = {
    str(row["metric"]): float(row["sample_std_all"]) for _, row in attack_resource_stats_df.iterrows()
}
attack_resource_pop_std: dict[str, float] = {
    str(row["metric"]): float(row["pop_std_all"]) for _, row in attack_resource_stats_df.iterrows()
}
winning_attack_resource_mean: dict[str, float] = {
    str(row["metric"]): float(row["mean_wins"]) for _, row in attack_resource_stats_df.iterrows()
}
winning_attack_resource_std: dict[str, float] = {
    str(row["metric"]): float(row["sample_std_wins"]) for _, row in attack_resource_stats_df.iterrows()
}

_recycle_arr = recycles_df["harvested_spice_eq"].to_numpy(dtype=np.float64)
recycle_spice_eq_mean, recycle_spice_eq_std, _ = compute_mean_and_std(_recycle_arr)

print("3. Attack Resource Loot & Net Combat Yield (`attack_resource_stats_df` via pandas & numpy)")
print(attack_resource_stats_df.round(2).to_string(index=False))
if len(recycles_df) > 0:
    print(
        f"\nSupplementary recycles_df ({len(recycles_df)} harvests): "
        f"mean={recycle_spice_eq_mean:,.2f} SpiceEq, sample_std={recycle_spice_eq_std:,.2f} SpiceEq"
    )

## 4. Mean & Standard Deviation of Attack Time (`pandas` & `numpy`)

To accurately simulate fleet-slot capacity (`1 + Fleet base b10` concurrent flights) and raid throughput per hour, this section extracts `numpy` arrays from `attacks_df` (`one_way_sec`, `round_trip_sec`) and `np.diff(attack_dispatch_ts_arr)`, computing vectorized means, sample standard deviations (`ddof=1`), population standard deviations (`ddof=0`), and medians inside **`attack_time_stats_df`**:

1. **Fleet Mission Flight Time (`one_way_sec` & `round_trip_sec`)**:
   - `one_way_sec` ($\mu_{t,\text{one-way}}, s_{t,\text{one-way}}$): Elapsed flight time from dispatch at origin colony to combat impact at target coordinates.
   - `round_trip_sec` ($\mu_{t,\text{round-trip}}, s_{t,\text{round-trip}}$): Full round-trip sortie duration ($2 \times t_{\text{one-way}}$), representing exact fleet-slot occupation time.
2. **Inter-Attack Dispatch Cadence (`inter_attack_sec`)**:
   - Positive consecutive dispatch intervals (`np.diff(attack_dispatch_ts_arr)`) measuring real wall-clock raid cadence.

In [ ]:
one_way_times_arr: np.ndarray = attacks_df["one_way_sec"].to_numpy(dtype=np.float64)
round_trip_times_arr: np.ndarray = attacks_df["round_trip_sec"].to_numpy(dtype=np.float64)

if attack_dispatch_ts_arr.size >= 2:
    _raw_deltas = np.diff(np.sort(attack_dispatch_ts_arr))
    inter_attack_intervals_arr: np.ndarray = _raw_deltas[_raw_deltas > 0.0]
else:
    inter_attack_intervals_arr = np.empty(0, dtype=np.float64)

one_way_times_sec: list[float] = one_way_times_arr.tolist()
round_trip_times_sec: list[float] = round_trip_times_arr.tolist()
inter_attack_intervals_sec: list[float] = inter_attack_intervals_arr.tolist()

attack_one_way_time_mean_sec, attack_one_way_time_std_sec, attack_one_way_time_pop_std_sec = (
    compute_mean_and_std(one_way_times_arr)
)
attack_time_mean_sec, attack_time_std_sec, attack_time_pop_std_sec = (
    compute_mean_and_std(round_trip_times_arr)
)
inter_attack_time_mean_sec, inter_attack_time_std_sec, inter_attack_time_pop_std_sec = (
    compute_mean_and_std(inter_attack_intervals_arr)
)

attack_time_stats_df = pd.DataFrame(
    [
        {
            "time_metric": "One-Way Flight Time (Dispatch -> Impact)",
            "samples": int(one_way_times_arr.size),
            "mean_sec": round(attack_one_way_time_mean_sec, 2),
            "mean_text": fmt_duration(attack_one_way_time_mean_sec),
            "sample_std_sec": round(attack_one_way_time_std_sec, 2),
            "sample_std_text": fmt_duration(attack_one_way_time_std_sec),
            "pop_std_sec": round(attack_one_way_time_pop_std_sec, 2),
            "median_sec": round(float(np.median(one_way_times_arr)) if one_way_times_arr.size > 0 else 0.0, 2),
        },
        {
            "time_metric": "Round-Trip Attack Time (Slot Occupation)",
            "samples": int(round_trip_times_arr.size),
            "mean_sec": round(attack_time_mean_sec, 2),
            "mean_text": fmt_duration(attack_time_mean_sec),
            "sample_std_sec": round(attack_time_std_sec, 2),
            "sample_std_text": fmt_duration(attack_time_std_sec),
            "pop_std_sec": round(attack_time_pop_std_sec, 2),
            "median_sec": round(float(np.median(round_trip_times_arr)) if round_trip_times_arr.size > 0 else 0.0, 2),
        },
        {
            "time_metric": "Inter-Attack Dispatch Interval",
            "samples": int(inter_attack_intervals_arr.size),
            "mean_sec": round(inter_attack_time_mean_sec, 2),
            "mean_text": fmt_duration(inter_attack_time_mean_sec),
            "sample_std_sec": round(inter_attack_time_std_sec, 2),
            "sample_std_text": fmt_duration(inter_attack_time_std_sec),
            "pop_std_sec": round(inter_attack_time_pop_std_sec, 2),
            "median_sec": round(float(np.median(inter_attack_intervals_arr)) if inter_attack_intervals_arr.size > 0 else 0.0, 2),
        },
    ]
)

print("4. Attack Time & Cadence Statistics (`attack_time_stats_df` via pandas & numpy)")
print(attack_time_stats_df.to_string(index=False))

## 5. Attack Success Rate (`pandas` & `numpy`)

This section evaluates the empirical **Attack Success Rate** and expected fleet-slot yield using vectorized `numpy` arrays (`attacks_df["won"].to_numpy(dtype=np.float64)`) and `pandas` `.groupby(...).agg(...)` rollups (`attack_class_stats_df`, `attack_target_stats_df`, `attack_summary_stats_df`):

$$\hat{p}_{\text{win}} = \frac{W}{N}, \qquad \text{SE}(\hat{p}_{\text{win}}) = \sqrt{\frac{\hat{p}_{\text{win}}(1 - \hat{p}_{\text{win}})}{N}}, \qquad \tilde{p}_{\text{win}} = \frac{W + 1}{N + 2}$$

- **`attack_success_rate` ($\hat{p}_{\text{win}}$)**, **`attack_success_rate_std_err`**, and **`attack_success_rate_smoothed` ($\tilde{p}_{\text{win}}$)**: Computed directly via `np.mean` / `np.sum` / `np.sqrt`.
- **`attack_class_stats_df` & `attack_target_stats_df`**: Grouped `pandas.DataFrame` aggregations across `target_class` and `target_coords` reporting sample counts, wins, losses, success rates, and `ddof=1` sample standard deviations of `spice_eq` and `net_spice_eq`.
- **`ATTACK_SIM_PARAMS` & `ATTACK_STATS_TABLES`**: Bundle scalar simulation parameters and `pandas` DataFrames for downstream full-planner simulations.

In [ ]:
_won_arr = attacks_df["won"].to_numpy(dtype=np.float64)
attack_attempts: int = int(_won_arr.size)
attack_wins: int = int(np.sum(_won_arr)) if attack_attempts > 0 else 0
attack_losses: int = attack_attempts - attack_wins

attack_success_rate: float = float(np.mean(_won_arr)) if attack_attempts > 0 else 0.0
attack_success_rate_std_err: float = (
    float(np.sqrt(attack_success_rate * (1.0 - attack_success_rate) / float(attack_attempts)))
    if attack_attempts > 0
    else 0.0
)
attack_success_rate_smoothed: float = float(attack_wins + 1) / float(attack_attempts + 2)

if len(attacks_df) > 0:
    attack_class_stats_df = (
        attacks_df.assign(won_num=attacks_df["won"].astype(np.float64))
        .groupby("target_class", as_index=False)
        .agg(
            attempts=("won_num", "count"),
            wins=("won_num", "sum"),
            success_rate=("won_num", "mean"),
            mean_spice_eq=("spice_eq", "mean"),
            sample_std_spice_eq=("spice_eq", "std"),
            mean_net_spice_eq=("net_spice_eq", "mean"),
            sample_std_net_spice_eq=("net_spice_eq", "std"),
        )
        .fillna(0.0)
        .sort_values("target_class")
        .reset_index(drop=True)
    )
    attack_class_stats_df["wins"] = attack_class_stats_df["wins"].astype(int)
    attack_class_stats_df["losses"] = attack_class_stats_df["attempts"] - attack_class_stats_df["wins"]

    attack_target_stats_df = (
        attacks_df.assign(won_num=attacks_df["won"].astype(np.float64))
        .groupby("target_coords", as_index=False)
        .agg(
            attempts=("won_num", "count"),
            wins=("won_num", "sum"),
            success_rate=("won_num", "mean"),
            mean_spice_eq=("spice_eq", "mean"),
            sample_std_spice_eq=("spice_eq", "std"),
            mean_net_spice_eq=("net_spice_eq", "mean"),
            sample_std_net_spice_eq=("net_spice_eq", "std"),
        )
        .fillna(0.0)
        .sort_values("target_coords")
        .reset_index(drop=True)
    )
    attack_target_stats_df["wins"] = attack_target_stats_df["wins"].astype(int)
    attack_target_stats_df["losses"] = attack_target_stats_df["attempts"] - attack_target_stats_df["wins"]
else:
    attack_class_stats_df = pd.DataFrame(
        columns=[
            "target_class",
            "attempts",
            "wins",
            "losses",
            "success_rate",
            "mean_spice_eq",
            "sample_std_spice_eq",
            "mean_net_spice_eq",
            "sample_std_net_spice_eq",
        ]
    )
    attack_target_stats_df = pd.DataFrame(
        columns=[
            "target_coords",
            "attempts",
            "wins",
            "losses",
            "success_rate",
            "mean_spice_eq",
            "sample_std_spice_eq",
            "mean_net_spice_eq",
            "sample_std_net_spice_eq",
        ]
    )

per_class_success_rate: dict[str, dict[str, Any]] = {
    str(row["target_class"]): {
        "attempts": int(row["attempts"]),
        "wins": int(row["wins"]),
        "losses": int(row["losses"]),
        "success_rate": float(row["success_rate"]),
        "mean_spice_eq": float(row["mean_spice_eq"]),
        "std_spice_eq": float(row["sample_std_spice_eq"]),
        "mean_net_spice_eq": float(row["mean_net_spice_eq"]),
    }
    for _, row in attack_class_stats_df.iterrows()
}

per_target_success_rate: dict[str, dict[str, Any]] = {
    str(row["target_coords"]): {
        "attempts": int(row["attempts"]),
        "wins": int(row["wins"]),
        "losses": int(row["losses"]),
        "success_rate": float(row["success_rate"]),
        "mean_spice_eq": float(row["mean_spice_eq"]),
        "std_spice_eq": float(row["sample_std_spice_eq"]),
        "mean_net_spice_eq": float(row["mean_net_spice_eq"]),
    }
    for _, row in attack_target_stats_df.iterrows()
}

expected_raids_per_slot_hr: float = (
    3600.0 / attack_time_mean_sec if attack_time_mean_sec > 0.0 else 0.0
)
expected_spice_eq_per_slot_hr: float = (
    expected_raids_per_slot_hr * attack_resource_mean["spice_eq"]
)
expected_net_spice_eq_per_slot_hr: float = (
    expected_raids_per_slot_hr * attack_resource_mean["net_spice_eq"]
)
max_empire_gross_spice_eq_hr: float = expected_spice_eq_per_slot_hr * float(fleet_slot_capacity)

attack_summary_stats_df = pd.DataFrame(
    [
        {"metric": "Total Attacks", "value": float(attack_attempts), "display": f"{attack_attempts} ({attack_wins} wins / {attack_losses} losses)"},
        {"metric": "Empirical Success Rate", "value": round(attack_success_rate, 6), "display": f"{attack_success_rate:.2%} (+/- {attack_success_rate_std_err:.2%} SE)"},
        {"metric": "Laplace-Smoothed Success Rate", "value": round(attack_success_rate_smoothed, 6), "display": f"{attack_success_rate_smoothed:.2%}"},
        {"metric": "Max Sorties per Fleet Slot / Hour", "value": round(expected_raids_per_slot_hr, 4), "display": f"{expected_raids_per_slot_hr:.2f} raids/hr (mean round-trip {fmt_duration(attack_time_mean_sec)})"},
        {"metric": "Gross Loot per Slot-Hour (SpiceEq)", "value": round(expected_spice_eq_per_slot_hr, 2), "display": f"{expected_spice_eq_per_slot_hr:,.1f} SpiceEq/hr"},
        {"metric": "Net Loot per Slot-Hour (SpiceEq)", "value": round(expected_net_spice_eq_per_slot_hr, 2), "display": f"{expected_net_spice_eq_per_slot_hr:+,.1f} SpiceEq/hr"},
        {"metric": f"Full Capacity Ceiling ({fleet_slot_capacity} Slots)", "value": round(max_empire_gross_spice_eq_hr, 2), "display": f"{max_empire_gross_spice_eq_hr:,.1f} gross SpiceEq/hr"},
    ]
)

ATTACK_STATS_TABLES: dict[str, pd.DataFrame] = {
    "attacks": attacks_df,
    "recycles": recycles_df,
    "attack_resource_stats": attack_resource_stats_df,
    "attack_time_stats": attack_time_stats_df,
    "attack_summary_stats": attack_summary_stats_df,
    "attack_class_stats": attack_class_stats_df,
    "attack_target_stats": attack_target_stats_df,
}

ATTACK_SIM_PARAMS: dict[str, Any] = {
    "attempts": attack_attempts,
    "wins": attack_wins,
    "losses": attack_losses,
    "success_rate": attack_success_rate,
    "success_rate_std_err": attack_success_rate_std_err,
    "success_rate_smoothed": attack_success_rate_smoothed,
    "fleet_slot_capacity": fleet_slot_capacity,
    "resource_mean": dict(attack_resource_mean),
    "resource_std": dict(attack_resource_std),
    "winning_resource_mean": dict(winning_attack_resource_mean),
    "winning_resource_std": dict(winning_attack_resource_std),
    "recycle_spice_eq_mean": recycle_spice_eq_mean,
    "recycle_spice_eq_std": recycle_spice_eq_std,
    "one_way_time_mean_sec": attack_one_way_time_mean_sec,
    "one_way_time_std_sec": attack_one_way_time_std_sec,
    "round_trip_time_mean_sec": attack_time_mean_sec,
    "round_trip_time_std_sec": attack_time_std_sec,
    "inter_attack_time_mean_sec": inter_attack_time_mean_sec,
    "inter_attack_time_std_sec": inter_attack_time_std_sec,
    "expected_raids_per_slot_hr": expected_raids_per_slot_hr,
    "expected_spice_eq_per_slot_hr": expected_spice_eq_per_slot_hr,
    "expected_net_spice_eq_per_slot_hr": expected_net_spice_eq_per_slot_hr,
    "max_empire_gross_spice_eq_hr": max_empire_gross_spice_eq_hr,
}

print("5. Attack Success Rate & Expected Fleet-Slot Yield (`attack_summary_stats_df` via pandas & numpy)")
print(attack_summary_stats_df[["metric", "display"]].to_string(index=False))
if not attack_class_stats_df.empty:
    print("\n[attack_class_stats_df — GroupBy SPY_TARGETS Class]")
    print(attack_class_stats_df.round(3).to_string(index=False))
if not attack_target_stats_df.empty:
    print("\n[attack_target_stats_df — GroupBy Target Coords]")
    print(attack_target_stats_df.round(3).to_string(index=False))

## 6. Game Parameters Catalog (`pandas` DataFrames) — Reference Context & Merchant Parity

This section builds the complete **`pandas` DataFrame parameter database** for SpaceInvasion Classic from `spaceinvasion.game_data`, normalizing all buildings (`b1`–`b21`), technologies (`r1`–`r15`), spaceships (`s1`–`s16`), and planetary defenses (`d1`–`d11`) for downstream strategy simulation.

First, we establish `GAME_PARAM_CONTEXT` (capturing the active universe `speedup_factor`, colony `temperature`, local construction/research/shipyard accelerators `b7` / `b8` / `b9` / `b17`, and global tech levels `r1`–`r15` from SQLite telemetry) alongside **`merchant_rates_df`**, which encodes fixed Merchant exchange parity (`6 Pig iron : 2.5 Metal : 3 Kryptonite : 1 Spice`) and the global ranking divisor (`1 point = 1,000 invested resources`).

In [ ]:
MAX_BUILDING_TABLE_LEVEL: int = 50
MAX_RESEARCH_TABLE_LEVEL: int = 30

_col_bld = telemetry_bundle.get("colony_buildings") or {}
_col_res = telemetry_bundle.get("colony_researches") or {}

GAME_PARAM_CONTEXT: dict[str, Any] = {
    "wiki_synced_at_utc": game_data.WIKI_SYNCED_AT_UTC,
    "speedup_factor": float(telemetry_bundle.get("universe_speedup_factor") or 5.0),
    "planet_temperature_c": int(telemetry_bundle.get("colony_temperature", 31)),
    "b7_dev_center": max(0, int(_col_bld.get("b7") or 0)),
    "b8_research_lab": max(0, int(_col_bld.get("b8") or 0)),
    "b9_weapons_factory": max(0, int(_col_bld.get("b9") or 0)),
    "b17_microsystem": max(0, int(_col_bld.get("b17") or 0)),
    "r1_mining_tech": max(0, int(_col_res.get("r1") or 0)),
    "r2_energy_tech": max(0, int(_col_res.get("r2") or 0)),
    "r4_rocket_engine": max(0, int(_col_res.get("r4") or 0)),
    "r5_diffusion_engine": max(0, int(_col_res.get("r5") or 0)),
    "r6_warp_engine": max(0, int(_col_res.get("r6") or 0)),
    "r8_weapons_tech": max(0, int(_col_res.get("r8") or 0)),
    "r9_shield_tech": max(0, int(_col_res.get("r9") or 0)),
    "r10_defense_systems": max(0, int(_col_res.get("r10") or 0)),
    "points_per_resource_divisor": float(POINTS_PER_RESOURCE_DIVISOR),
}

BUILDING_TIME_DIVISOR: float = max(
    1.0,
    float(GAME_PARAM_CONTEXT["b7_dev_center"] + 1)
    * float(2 ** min(GAME_PARAM_CONTEXT["b17_microsystem"], 40))
    * float(GAME_PARAM_CONTEXT["speedup_factor"]),
)
RESEARCH_TIME_DIVISOR: float = max(
    1.0,
    float(GAME_PARAM_CONTEXT["b8_research_lab"] + 1)
    * float(GAME_PARAM_CONTEXT["speedup_factor"]),
)
SHIPYARD_TIME_DIVISOR: float = max(
    1.0,
    float(max(1, GAME_PARAM_CONTEXT["b9_weapons_factory"]))
    * float(2 ** min(GAME_PARAM_CONTEXT["b17_microsystem"], 40))
    * float(GAME_PARAM_CONTEXT["speedup_factor"]),
)

_RESOURCE_NOTES: dict[str, str] = {
    "pig_iron": "Direct planetary mine extraction (+2%/lvl r1); feedstock for Blast furnace (2:1)",
    "metal": "Smelted 2:1 from Pig iron inside Blast furnace (b2); NOT boosted by r1",
    "kryptonite": "Direct planetary mine extraction (+2%/lvl r1); heavy tech & warship input",
    "spice": "Temperature-interpolated output (higher on colder colonies) +2%/lvl r1; fleet & scan fuel",
}

merchant_rates_df = pd.DataFrame(
    [
        {
            "resource": rk,
            "units_per_1_spice": float(game_data.MERCHANT_RATES_IN_SPICE[rk]),
            "spice_eq_per_unit": round(1.0 / float(game_data.MERCHANT_RATES_IN_SPICE[rk]), 6),
            "points_per_unit": 1.0 / float(POINTS_PER_RESOURCE_DIVISOR),
            "mine_building_key": f"b{idx + 1}",
            "boosted_by_mining_tech_r1": rk != "metal",
            "notes": _RESOURCE_NOTES[rk],
        }
        for idx, rk in enumerate(RESOURCE_KEYS)
    ]
)

print(
    f"Game Parameter Context: speed={GAME_PARAM_CONTEXT['speedup_factor']:.1f}x, "
    f"temp={GAME_PARAM_CONTEXT['planet_temperature_c']}°C | "
    f"Divisors -> Build={BUILDING_TIME_DIVISOR:.1f}x (b7 L{GAME_PARAM_CONTEXT['b7_dev_center']}, b17 L{GAME_PARAM_CONTEXT['b17_microsystem']}), "
    f"Research={RESEARCH_TIME_DIVISOR:.1f}x (b8 L{GAME_PARAM_CONTEXT['b8_research_lab']}), "
    f"Shipyard={SHIPYARD_TIME_DIVISOR:.1f}x (b9 L{GAME_PARAM_CONTEXT['b9_weapons_factory']})"
)
print("\n[merchant_rates_df]")
print(merchant_rates_df.to_string(index=False))

### 6.1 Buildings Parameters (`b1`–`b21`) — Costs, Build Times, Production, Energy & Storage (`buildings_catalog_df`, `buildings_levels_df`)

This component materializes two complementary `pandas.DataFrame` tables covering all **21 SpaceInvasion Classic buildings**:

1. **`buildings_catalog_df` (`21` rows, `b1`–`b21`)**: Master metadata per building including `category`, `asteroid_ok`, `max_table_level`, `cost_scale_factor`, `L1` resource bundle (`pig_iron`, `metal`, `kryptonite`, `spice`, `l1_cost_spice_eq`), `L1` base and colony-effective build times, tech-tree `prerequisites`, and `primary_effect`.
2. **`buildings_levels_df` (`L1..L50` per building)**: Exact level-by-level progression tracking:
   - **Costs & Points**: `pig_iron`, `metal`, `kryptonite`, `spice`, `cost_spice_eq`, and `points_gain` (`cost_sum / 1,000`).
   - **Construction Duration**: `base_time_s` (`1x` Wiki baseline) and `effective_build_time_s = base_time_s / ((b7 + 1) * 2^b17 * speedup_factor)`.
   - **Hourly Production & Feedstock**: `hourly_prod_1x`, `hourly_prod_effective` (`+2%/lvl` `r1` on `b1`/`b3`/`b4`, temperature-interpolated for `b4` *Spice mine*; unboosted for `b2` *Blast furnace*), `pig_iron_consumed_hr_effective` (`2:1` smelting feedstock on `b2`), `spice_consumed_hr` (`b6` *Thermosolar* fuel), marginal `delta_net_spice_eq_hr`, and incremental `marginal_roi_hours`.
   - **Energy, Storage & Accelerators**: `energy_at_level_base`, `energy_at_level_effective` (`+1%/lvl` `r2` on `b5`/`b6`), `energy_delta_from_prev`, warehouse `storage_capacity` (`100,000 * 2^L` on `b11`–`b14`), and `time_reduction_multiplier` (`1/(L+1)` on `b7`, `1/L` on `b9`, `0.5^L` on `b17`).

In [ ]:
_BUILDING_META: dict[int, tuple[str, float, str]] = {
    1: ("resource_mine", 1.51, "Mines Pig iron (+2%/lvl r1); consumes energy"),
    2: ("smelter", 1.51, "Smelts Pig iron -> Metal at 2:1 loss (NOT boosted by r1); consumes energy"),
    3: ("resource_mine", 1.51, "Mines Kryptonite (+2%/lvl r1); consumes energy"),
    4: ("resource_mine", 1.51, "Mines Spice on planet-temperature curve (+2%/lvl r1); consumes energy"),
    5: ("power_plant", 1.51, "Fuel-free planetary power (+1%/lvl r2); 1 field/level"),
    6: ("power_plant", 1.51, "Spice-burning planetary power (+1%/lvl r2); forbidden on normal planets when forbid_planetary_solar=True"),
    7: ("accelerator", 2.00, "Cuts planetary building construction time by 1 / (L + 1); gates b8, b9, b17, b19"),
    8: ("research_infra", 2.00, "Unlocks & accelerates global research time by 1 / (L + 1)"),
    9: ("shipyard", 2.00, "Unlocks ships/defenses & cuts unit build time by 1 / max(1, L)"),
    10: ("fleet_infra", 2.00, "Grants 1 + L concurrent fleet flight slots & allied fleet support"),
    11: ("warehouse", 2.00, "Pig iron storage capacity 100,000 * 2^L (prevents 5% overflow penalty)"),
    12: ("warehouse", 2.00, "Metal storage capacity 100,000 * 2^L"),
    13: ("warehouse", 2.00, "Kryptonite storage capacity 100,000 * 2^L"),
    14: ("warehouse", 2.00, "Spice storage capacity 100,000 * 2^L"),
    15: ("strategic_weapon", 2.00, "Stores surplus energy to fire remote particle beams at enemy defenses"),
    16: ("strategic_defense", 2.00, "Absorbs incoming Particle cannon beams (46% / 31% / 23% priority split)"),
    17: ("accelerator", 2.00, "Halves all building & shipyard build times each level (0.5^L)"),
    18: ("fleet_infra", 2.00, "Instant zero-flight-time fleet jump between charged Teleporters"),
    19: ("asteroid_sensor", 2.00, "Asteroid-exclusive sensor scanning enemy fleet flights (5,000 Spice/scan; Escape flights hidden)"),
    20: ("strategic_defense", 1.08, "Cuts defense dig-in time (60m - 1m/L) & adds L40+ particle fog"),
    21: ("endgame_synth", 1.44, "Grants +1 Synth Point per level (up to L30, highest planet only)"),
}

_PRODUCED_RES_BY_BID: dict[int, str] = {
    1: "pig_iron",
    2: "metal",
    3: "kryptonite",
    4: "spice",
}

_speedup_f = float(GAME_PARAM_CONTEXT["speedup_factor"])
_temp_c = int(GAME_PARAM_CONTEXT["planet_temperature_c"])
_r1_lvl = int(GAME_PARAM_CONTEXT["r1_mining_tech"])
_r2_lvl = int(GAME_PARAM_CONTEXT["r2_energy_tech"])
_energy_mult = 1.0 + 0.01 * float(_r2_lvl)

_b_cat_rows: list[dict[str, Any]] = []
_b_lvl_rows: list[dict[str, Any]] = []

for _bid in sorted(game_data.BUILDING_BY_ID.keys()):
    _info = game_data.BUILDING_BY_ID[_bid]
    _bkey = f"b{_bid}"
    _cat, _scale, _effect = _BUILDING_META.get(_bid, ("other", 2.0, ""))
    _max_tbl = int(_info.get("max_table_level") or len(_info.get("levels") or {}))
    _lvls_map = _info.get("levels") or {}
    _mono_times = build_monotonic_base_times(_lvls_map, _max_tbl)
    _l1_cost = game_data.building_cost(_bid, 1)
    _l1_sp = float(game_data.spice_equivalent(_l1_cost))
    _l1_bt = float(_mono_times.get(1, 0.0))

    _b_cat_rows.append(
        {
            "building_id": _bid,
            "key": _bkey,
            "slug": str(_info["slug"]),
            "name": str(_info["name"]),
            "category": _cat,
            "asteroid_ok": bool(_info.get("asteroid_ok", False)),
            "max_table_level": _max_tbl,
            "cost_scale_factor": _scale,
            "l1_pig_iron": int(_l1_cost["pig_iron"]),
            "l1_metal": int(_l1_cost["metal"]),
            "l1_kryptonite": int(_l1_cost["kryptonite"]),
            "l1_spice": int(_l1_cost["spice"]),
            "l1_cost_spice_eq": round(_l1_sp, 2),
            "l1_base_time_s": round(_l1_bt, 2),
            "l1_effective_time_s": round(_l1_bt / BUILDING_TIME_DIVISOR, 2),
            "prerequisites": format_prerequisites(_info.get("requirements") or []),
            "primary_effect": _effect,
        }
    )

    _prev_net_sp_hr = 0.0
    _prev_energy_eff = 0.0
    _sim_cap = min(_max_tbl, MAX_BUILDING_TABLE_LEVEL)
    for _lvl in range(1, _sim_cap + 1):
        _row = _lvls_map.get(_lvl) or {}
        _cost = game_data.building_cost(_bid, _lvl)
        _cost_sp = float(game_data.spice_equivalent(_cost))
        _pts_gain = float(sum(_cost.values())) / float(POINTS_PER_RESOURCE_DIVISOR)
        _bt_s = float(_mono_times.get(_lvl, 0.0))
        _eff_t_s = _bt_s / BUILDING_TIME_DIVISOR

        _prod_res = _PRODUCED_RES_BY_BID.get(_bid, "none")
        if _bid in (1, 2, 3, 4):
            _prod_1x = float(
                game_data.mine_production_per_hour(_bid, _lvl, temperature=_temp_c, mining_tech_level=0)
            )
            _prod_eff = float(
                game_data.mine_production_per_hour(_bid, _lvl, temperature=_temp_c, mining_tech_level=_r1_lvl)
            ) * _speedup_f
        else:
            _prod_1x = 0.0
            _prod_eff = 0.0

        _pig_used_1x = float(game_data.blast_furnace_pig_iron_used(_lvl)) if _bid == 2 else 0.0
        _pig_used_eff = _pig_used_1x * _speedup_f
        _spice_used_hr = float(_row.get("spice_used") or 0.0) if _bid == 6 else 0.0

        _gross_sp_hr = (
            _prod_eff / float(game_data.MERCHANT_RATES_IN_SPICE[_prod_res])
            if _prod_res in game_data.MERCHANT_RATES_IN_SPICE
            else 0.0
        )
        _feedstock_sp_hr = (
            _pig_used_eff / float(game_data.MERCHANT_RATES_IN_SPICE["pig_iron"])
            + _spice_used_hr / float(game_data.MERCHANT_RATES_IN_SPICE["spice"])
        )
        _net_sp_hr = _gross_sp_hr - _feedstock_sp_hr
        _delta_net_sp_hr = _net_sp_hr - _prev_net_sp_hr
        _roi_h = (_cost_sp / _delta_net_sp_hr) if _delta_net_sp_hr > 0.0 else float("inf")
        _prev_net_sp_hr = _net_sp_hr

        _e_base = float(_row.get("energy") or 0.0)
        _e_eff = (_e_base * _energy_mult) if _e_base > 0.0 else _e_base
        _e_delta = _e_eff - _prev_energy_eff
        _prev_energy_eff = _e_eff

        _storage_cap = int(_row.get("storage_capacity") or (100_000 * (2**_lvl))) if _bid in (11, 12, 13, 14) else 0
        if _bid == 7:
            _time_mult = 1.0 / float(_lvl + 1)
        elif _bid == 8:
            _time_mult = 1.0 / float(_lvl + 1)
        elif _bid == 9:
            _time_mult = 1.0 / float(max(1, _lvl))
        elif _bid == 17:
            _time_mult = 0.5 ** float(_lvl)
        else:
            _time_mult = 1.0

        _b_lvl_rows.append(
            {
                "building_id": _bid,
                "key": _bkey,
                "slug": str(_info["slug"]),
                "name": str(_info["name"]),
                "category": _cat,
                "level": _lvl,
                "pig_iron": int(_cost["pig_iron"]),
                "metal": int(_cost["metal"]),
                "kryptonite": int(_cost["kryptonite"]),
                "spice": int(_cost["spice"]),
                "cost_spice_eq": round(_cost_sp, 2),
                "points_gain": round(_pts_gain, 3),
                "base_time_s": round(_bt_s, 2),
                "effective_build_time_s": round(_eff_t_s, 2),
                "effective_build_time_text": fmt_duration(_eff_t_s),
                "produced_resource": _prod_res,
                "hourly_prod_1x": round(_prod_1x, 2),
                "hourly_prod_effective": round(_prod_eff, 2),
                "pig_iron_consumed_hr_effective": round(_pig_used_eff, 2),
                "spice_consumed_hr": round(_spice_used_hr, 2),
                "net_spice_eq_hr_effective": round(_net_sp_hr, 2),
                "delta_net_spice_eq_hr": round(_delta_net_sp_hr, 2),
                "marginal_roi_hours": round(_roi_h, 2) if math.isfinite(_roi_h) else None,
                "energy_at_level_base": round(_e_base, 2),
                "energy_at_level_effective": round(_e_eff, 2),
                "energy_delta_from_prev": round(_e_delta, 2),
                "storage_capacity": _storage_cap,
                "time_reduction_multiplier": _time_mult,
            }
        )

buildings_catalog_df = pd.DataFrame(_b_cat_rows)
buildings_levels_df = pd.DataFrame(_b_lvl_rows)

print(
    f"Built buildings_catalog_df ({buildings_catalog_df.shape[0]} rows x {buildings_catalog_df.shape[1]} cols) "
    f"and buildings_levels_df ({buildings_levels_df.shape[0]} rows x {buildings_levels_df.shape[1]} cols)"
)
print("\n[buildings_catalog_df — All 21 Buildings]")
print(
    buildings_catalog_df[
        [
            "key",
            "name",
            "category",
            "asteroid_ok",
            "cost_scale_factor",
            "l1_cost_spice_eq",
            "l1_effective_time_s",
            "prerequisites",
        ]
    ].to_string(index=False)
)
print("\n[buildings_levels_df — Economy Checkpoints (b1..b5 at L1, L10, L20)]")
print(
    buildings_levels_df[
        buildings_levels_df["building_id"].isin([1, 2, 3, 4, 5])
        & buildings_levels_df["level"].isin([1, 10, 20])
    ][
        [
            "key",
            "name",
            "level",
            "cost_spice_eq",
            "effective_build_time_text",
            "hourly_prod_effective",
            "net_spice_eq_hr_effective",
            "marginal_roi_hours",
            "energy_at_level_effective",
        ]
    ].to_string(index=False)
)

### 6.2 Technologies Parameters (`r1`–`r15`) — Costs, Research Times & Quantitative Effects (`technologies_catalog_df`, `technologies_levels_df`)

All SpaceInvasion Classic technologies apply **account-wide** across every planet, asteroid, and fleet upon completion, scaling resource costs and base research times by exact doubling (`2.0^(L - 1)`) and accelerating by the local planet's *Research lab* (`b8`) via `effective_research_time_s = base_time_s / ((b8 + 1) * speedup_factor)`.

This component builds two `pandas.DataFrame` tables:

1. **`technologies_catalog_df` (`15` rows, `r1`–`r15`)**: Encodes `domain`, `max_table_level`, `L1` resource costs & `l1_cost_spice_eq`, `L1` base & effective research durations, `effect_metric`, exact **`per_level_bonus_pct`** (`+2%/lvl` `r1` mines excluding `b2`, `+1%/lvl` `r2` global energy, `+10%/lvl` `r4`/`r5`/`r6` ship speed by engine class, `+10%/lvl` `r8` attack / `r9` shield / `r10` hull), `applies_to`, `prerequisites`, and `strategic_effect`.
2. **`technologies_levels_df` (`L1..L30` per research)**: Level-by-level progression with `pig_iron..spice`, `cost_spice_eq`, `points_gain`, `base_time_s`, `effective_research_time_s`, cumulative `bonus_pct_total`, `multiplier_at_level`, and human-readable `effect_at_level`.

In [ ]:
_TECH_EFFECT_SPECS: dict[str, dict[str, Any]] = {
    "r1": {
        "domain": "economy",
        "effect_metric": "mine_output_bonus",
        "per_level_bonus_pct": 2.0,
        "applies_to": "b1 Pig iron, b3 Kryptonite, b4 Spice across all planets (NOT b2 Blast furnace)",
        "strategic_effect": "+2%/lvl global Pig iron, Kryptonite & Spice mine output",
    },
    "r2": {
        "domain": "economy",
        "effect_metric": "global_energy_bonus",
        "per_level_bonus_pct": 1.0,
        "applies_to": "b5 Fusion plant, b6 Thermosolar plant, s9 Solar cells across all planets",
        "strategic_effect": "+1%/lvl global energy output; gates b9, b6, r1, shields & turrets",
    },
    "r3": {
        "domain": "propulsion",
        "effect_metric": "engine_tier_unlock",
        "per_level_bonus_pct": 0.0,
        "applies_to": "Unlocks Rocket (L2), Diffusion (L6), and Warp (L8) drive trees",
        "strategic_effect": "Core propulsion gate (L2 -> r4, L6 -> r5, L8 -> r6)",
    },
    "r4": {
        "domain": "propulsion",
        "effect_metric": "rocket_ship_speed_bonus",
        "per_level_bonus_pct": 10.0,
        "applies_to": "s1 Small transport, s2 Large transport, s4 Small fighter, s5 Warship, s8 Colony ship, s10 Spy probe",
        "strategic_effect": "+10%/lvl flight speed on Rocket-engine ships",
    },
    "r5": {
        "domain": "propulsion",
        "effect_metric": "diffusion_ship_speed_bonus",
        "per_level_bonus_pct": 10.0,
        "applies_to": "s3 Transmitter, s6 Starcruiser, s7 Phoenix, s11 Recycler",
        "strategic_effect": "+10%/lvl flight speed on Diffusion-engine fast-raid & recycler fleets",
    },
    "r6": {
        "domain": "propulsion",
        "effect_metric": "warp_ship_speed_bonus",
        "per_level_bonus_pct": 10.0,
        "applies_to": "s13 Battleship, s14 Stealth bomber, s15 Destroyer, s16 Imperial starbase",
        "strategic_effect": "+10%/lvl flight speed on heavy Warp-engine capital wings",
    },
    "r7": {
        "domain": "intelligence",
        "effect_metric": "espionage_and_stealth",
        "per_level_bonus_pct": 0.0,
        "applies_to": "s10 Spy probe intel depth, counter-espionage, s14 Stealth bomber late detection",
        "strategic_effect": "Expands spy report tiers (fleets/defense/buildings/research) & Stealth concealment",
    },
    "r8": {
        "domain": "combat",
        "effect_metric": "unit_attack_bonus",
        "per_level_bonus_pct": 10.0,
        "applies_to": "All ships (s1..s16) and planetary defenses (d1..d11)",
        "strategic_effect": "+10%/lvl weapon attack power on all combat units",
    },
    "r9": {
        "domain": "combat",
        "effect_metric": "unit_shield_bonus",
        "per_level_bonus_pct": 10.0,
        "applies_to": "All ships (s1..s16) and planetary defenses (d1..d11)",
        "strategic_effect": "+10%/lvl shield absorption capacity; gates Shield domes & r12",
    },
    "r10": {
        "domain": "combat",
        "effect_metric": "unit_hull_bonus",
        "per_level_bonus_pct": 10.0,
        "applies_to": "All ships (s1..s16) and planetary defenses (d1..d11)",
        "strategic_effect": "+10%/lvl structural hull/armor points; gates s7 Phoenix (L5)",
    },
    "r11": {
        "domain": "defense_unlock",
        "effect_metric": "turret_tier_unlock",
        "per_level_bonus_pct": 0.0,
        "applies_to": "Ion, Heavy graviton, Positron, Interval, Laser, and Solar cannons",
        "strategic_effect": "Unlocks mid-to-heavy planetary turret tiers (L4..L10)",
    },
    "r12": {
        "domain": "strategic_weapon",
        "effect_metric": "particle_tech_unlock",
        "per_level_bonus_pct": 0.0,
        "applies_to": "b16 Particle shield (L2), d6 Interval cannon (L4), b15 Particle cannon (L5)",
        "strategic_effect": "Gates planetary Particle shield, Interval cannon & Particle cannon",
    },
    "r13": {
        "domain": "defense_unlock",
        "effect_metric": "graviton_unlock",
        "per_level_bonus_pct": 0.0,
        "applies_to": "Light/Heavy graviton cannons & r15 Black hole research (L10)",
        "strategic_effect": "Unlocks Graviton turrets & gates Black hole research",
    },
    "r14": {
        "domain": "logistics",
        "effect_metric": "teleporter_cooldown_and_unlock",
        "per_level_bonus_pct": 0.0,
        "applies_to": "b18 Teleporter (unlocked at L6) cooldown & energy efficiency",
        "strategic_effect": "Unlocks b18 Teleporter (L6) & reduces jump cooldown/energy",
    },
    "r15": {
        "domain": "endgame_warfare",
        "effect_metric": "scanner_jam_duration_hours",
        "per_level_bonus_pct": 0.0,
        "applies_to": "s16 Imperial starbase unlock (L2) & system scanner-blindness (4h + 1h * L)",
        "strategic_effect": "Unlocks s16 Imperial starbase (L2) & extends scanner jam to (4 + L) hours",
    },
}

_r_by_num: list[tuple[int, str, dict[str, Any]]] = []
for _slug, _rinfo in game_data.RESEARCHES.items():
    _rkey = _rinfo.get("id")
    if isinstance(_rkey, str) and _rkey.startswith("r") and _rkey[1:].isdigit():
        _r_by_num.append((int(_rkey[1:]), _rkey, _rinfo))
_r_by_num.sort(key=lambda item: item[0])

_t_cat_rows: list[dict[str, Any]] = []
_t_lvl_rows: list[dict[str, Any]] = []

for _rnum, _rkey, _rinfo in _r_by_num:
    _spec = _TECH_EFFECT_SPECS.get(_rkey, {})
    _lvls_map = _rinfo.get("levels") or {}
    _max_tbl = int(_rinfo.get("max_table_level") or len(_lvls_map))
    _mono_times = build_monotonic_base_times(_lvls_map, _max_tbl)
    _l1_cost = dict((_lvls_map.get(1) or {}).get("cost") or _rinfo.get("base_cost") or {})
    for _rk in RESOURCE_KEYS:
        _l1_cost.setdefault(_rk, 0)
    _l1_sp = float(game_data.spice_equivalent(_l1_cost))
    _l1_bt = float(_mono_times.get(1, 0.0))
    _bonus_pct = float(_spec.get("per_level_bonus_pct") or 0.0)
    _cur_lvl = max(0, int((telemetry_bundle.get("colony_researches") or {}).get(_rkey) or 0))

    _t_cat_rows.append(
        {
            "research_id": _rnum,
            "key": _rkey,
            "slug": str(_rinfo["slug"]),
            "name": str(_rinfo["name"]),
            "domain": str(_spec.get("domain", "general")),
            "current_level": _cur_lvl,
            "max_table_level": _max_tbl,
            "cost_scale_factor": 2.0,
            "l1_pig_iron": int(_l1_cost["pig_iron"]),
            "l1_metal": int(_l1_cost["metal"]),
            "l1_kryptonite": int(_l1_cost["kryptonite"]),
            "l1_spice": int(_l1_cost["spice"]),
            "l1_cost_spice_eq": round(_l1_sp, 2),
            "l1_base_time_s": round(_l1_bt, 2),
            "l1_effective_time_s": round(_l1_bt / RESEARCH_TIME_DIVISOR, 2),
            "effect_metric": str(_spec.get("effect_metric", "unlock")),
            "per_level_bonus_pct": _bonus_pct,
            "applies_to": str(_spec.get("applies_to", "")),
            "prerequisites": format_prerequisites(_rinfo.get("requirements") or []),
            "strategic_effect": str(_spec.get("strategic_effect", "")),
        }
    )

    _sim_cap = min(_max_tbl, MAX_RESEARCH_TABLE_LEVEL)
    for _lvl in range(1, _sim_cap + 1):
        _row = _lvls_map.get(_lvl) or {}
        _cost = dict(_row.get("cost") or {})
        for _rk in RESOURCE_KEYS:
            _cost.setdefault(_rk, 0)
        _cost_sp = float(game_data.spice_equivalent(_cost))
        _pts_gain = float(sum(int(_cost[k]) for k in RESOURCE_KEYS)) / float(POINTS_PER_RESOURCE_DIVISOR)
        _bt_s = float(_mono_times.get(_lvl, 0.0))
        _eff_t_s = _bt_s / RESEARCH_TIME_DIVISOR
        _tot_pct = _bonus_pct * float(_lvl)
        _mult = 1.0 + (_tot_pct / 100.0)
        if _bonus_pct > 0.0:
            _eff_txt = f"+{_tot_pct:.0f}% ({_mult:.2f}x)"
        elif _rkey == "r15":
            _eff_txt = f"Scanner jam {4 + _lvl}h" + (" + Starbase unlocked" if _lvl >= 2 else "")
        else:
            _eff_txt = f"Tier L{_lvl} unlock / capability"

        _t_lvl_rows.append(
            {
                "research_id": _rnum,
                "key": _rkey,
                "slug": str(_rinfo["slug"]),
                "name": str(_rinfo["name"]),
                "domain": str(_spec.get("domain", "general")),
                "level": _lvl,
                "pig_iron": int(_cost["pig_iron"]),
                "metal": int(_cost["metal"]),
                "kryptonite": int(_cost["kryptonite"]),
                "spice": int(_cost["spice"]),
                "cost_spice_eq": round(_cost_sp, 2),
                "points_gain": round(_pts_gain, 3),
                "base_time_s": round(_bt_s, 2),
                "effective_research_time_s": round(_eff_t_s, 2),
                "effective_research_time_text": fmt_duration(_eff_t_s),
                "bonus_pct_total": round(_tot_pct, 2),
                "multiplier_at_level": round(_mult, 4),
                "effect_at_level": _eff_txt,
            }
        )

technologies_catalog_df = pd.DataFrame(_t_cat_rows)
technologies_levels_df = pd.DataFrame(_t_lvl_rows)

print(
    f"Built technologies_catalog_df ({technologies_catalog_df.shape[0]} rows x {technologies_catalog_df.shape[1]} cols) "
    f"and technologies_levels_df ({technologies_levels_df.shape[0]} rows x {technologies_levels_df.shape[1]} cols)"
)
print("\n[technologies_catalog_df — All 15 Researches & Effects]")
print(
    technologies_catalog_df[
        [
            "key",
            "name",
            "domain",
            "current_level",
            "l1_cost_spice_eq",
            "l1_effective_time_s",
            "per_level_bonus_pct",
            "prerequisites",
            "strategic_effect",
        ]
    ].to_string(index=False)
)

### 6.3 Spaceships Parameters (`s1`–`s16`) — Costs, Build Times, Combat, Propulsion, Cargo & Orbital Energy (`ships_df`)

This component builds **`ships_df`** (`15` standard SpaceInvasion Classic ship classes indexed via `SHIP_SLUG_BY_KEY`), computing both **base Wiki parameters** and **active-empire effective parameters**:

- **Unit Costs & Ranking Value**: `pig_iron`, `metal`, `kryptonite`, `spice`, `cost_spice_eq`, and `points_value` (`cost_sum / 1,000`).
- **Shipyard Build Time & Throughput**: `base_build_time_s`, `effective_build_time_s = base_build_time_s / (max(1, b9) * 2^b17 * speedup_factor)`, and `units_per_hour_effective`.
- **Zero-Rapidfire (`1:1`) Combat Stats**: Base and tech-boosted `hull` (`+10%/lvl r10`), `shield` (`+10%/lvl r9`), and `attack` (`+10%/lvl r8`), plus `attack_per_spice_eq` and `hull_per_spice_eq`.
- **Propulsion, Logistics & Orbital Energy**: Governing `engine` (`Rocket -> r4`, `Diffusion -> r5`, `Warp -> r6`), `speed_base`, `speed_effective` (`+10%/lvl`), `cargo`, `fuel`, `cargo_per_spice_eq`, orbital `energy_output_effective` (`s9` *Solar cells*, `0` fields used), and tech-tree `prerequisites`.

In [ ]:
_SHIP_ROLE_BY_KEY: dict[str, str] = {
    "s1": "cargo_transport",
    "s2": "cargo_transport",
    "s3": "armed_heavy_hauler",
    "s4": "fodder_interceptor",
    "s5": "early_raider",
    "s6": "cruiser_raider",
    "s7": "fast_strike_cruiser",
    "s8": "colonizer",
    "s9": "orbital_power_satellite",
    "s10": "espionage_probe",
    "s11": "debris_recycler",
    "s13": "capital_warship",
    "s14": "first_strike_bomber",
    "s15": "heavy_destroyer",
    "s16": "supercapital_starbase",
}

_ENGINE_TECH_KEY_MAP: dict[str, str] = {
    "rocket": "r4",
    "diffusion": "r5",
    "warp": "r6",
}

_atk_mult = 1.0 + 0.10 * float(GAME_PARAM_CONTEXT["r8_weapons_tech"])
_shd_mult = 1.0 + 0.10 * float(GAME_PARAM_CONTEXT["r9_shield_tech"])
_hul_mult = 1.0 + 0.10 * float(GAME_PARAM_CONTEXT["r10_defense_systems"])
_eng_mult = 1.0 + 0.01 * float(GAME_PARAM_CONTEXT["r2_energy_tech"])
_planet_temp_c = float(GAME_PARAM_CONTEXT["planet_temperature_c"])

_ship_rows: list[dict[str, Any]] = []
for _skey in sorted(SHIP_SLUG_BY_KEY.keys(), key=lambda k: int(k[1:])):
    _sid = int(_skey[1:])
    _slug = SHIP_SLUG_BY_KEY[_skey]
    _sinfo = game_data.SHIPS[_slug]
    _cost = dict(_sinfo.get("base_cost") or {})
    for _rk in RESOURCE_KEYS:
        _cost.setdefault(_rk, 0)
    _cost_sp = float(game_data.spice_equivalent(_cost))
    _pts_val = float(sum(int(_cost[k]) for k in RESOURCE_KEYS)) / float(POINTS_PER_RESOURCE_DIVISOR)
    _base_bt = float(_sinfo.get("base_build_time_s") or 0.0)
    _eff_bt = _base_bt / SHIPYARD_TIME_DIVISOR
    _units_hr = (3600.0 / _eff_bt) if _eff_bt > 0.0 else 0.0

    _st = _sinfo.get("stats") or {}
    _hull_b = int(_st.get("hull") or 0)
    _shld_b = int(_st.get("shield") or 0)
    _atk_b = int(_st.get("attack") or 0)
    _spd_b = int(_st.get("speed") or 0)
    _cargo = int(_st.get("cargo") or 0)
    _fuel = int(_st.get("fuel") or 0)
    _engine_raw = str(_st.get("engine") or ("Stationary" if _skey == "s9" else "Unknown"))

    _eng_tech_key = "none"
    for _prefix, _tk in _ENGINE_TECH_KEY_MAP.items():
        if _prefix in _engine_raw.lower():
            _eng_tech_key = _tk
            break
    _eng_lvl = int(
        GAME_PARAM_CONTEXT.get(
            {
                "r4": "r4_rocket_engine",
                "r5": "r5_diffusion_engine",
                "r6": "r6_warp_engine",
            }.get(_eng_tech_key, ""),
            0,
        )
    )
    _spd_eff = float(_spd_b) * (1.0 + 0.10 * float(_eng_lvl))

    _solar_base = solar_cell_energy_per_unit(_planet_temp_c) if _skey == "s9" else 0.0
    _solar_eff = _solar_base * _eng_mult

    _ship_rows.append(
        {
            "ship_id": _sid,
            "key": _skey,
            "slug": _slug,
            "name": str(_sinfo["name"]),
            "role": _SHIP_ROLE_BY_KEY.get(_skey, "fleet"),
            "pig_iron": int(_cost["pig_iron"]),
            "metal": int(_cost["metal"]),
            "kryptonite": int(_cost["kryptonite"]),
            "spice": int(_cost["spice"]),
            "cost_spice_eq": round(_cost_sp, 2),
            "points_value": round(_pts_val, 3),
            "base_build_time_s": round(_base_bt, 2),
            "effective_build_time_s": round(_eff_bt, 2),
            "effective_build_time_text": fmt_duration(_eff_bt),
            "units_per_hour_effective": round(_units_hr, 2),
            "hull_base": _hull_b,
            "shield_base": _shld_b,
            "attack_base": _atk_b,
            "hull_effective": round(float(_hull_b) * _hul_mult, 1),
            "shield_effective": round(float(_shld_b) * _shd_mult, 1),
            "attack_effective": round(float(_atk_b) * _atk_mult, 1),
            "engine": _engine_raw,
            "engine_tech_key": _eng_tech_key,
            "speed_base": _spd_b,
            "speed_effective": round(_spd_eff, 1),
            "cargo": _cargo,
            "fuel": _fuel,
            "energy_output_base": round(_solar_base, 2),
            "energy_output_effective": round(_solar_eff, 2),
            "fields_used": 0,
            "cargo_per_spice_eq": round(float(_cargo) / _cost_sp, 3) if _cost_sp > 0.0 else 0.0,
            "attack_per_spice_eq": round((float(_atk_b) * _atk_mult) / _cost_sp, 4) if _cost_sp > 0.0 else 0.0,
            "hull_per_spice_eq": round((float(_hull_b) * _hul_mult) / _cost_sp, 2) if _cost_sp > 0.0 else 0.0,
            "prerequisites": format_prerequisites(_sinfo.get("requirements") or []),
        }
    )

ships_df = pd.DataFrame(_ship_rows)

print(f"Built ships_df ({ships_df.shape[0]} rows x {ships_df.shape[1]} cols)")
print("\n[ships_df — All 15 Spaceships]")
print(
    ships_df[
        [
            "key",
            "name",
            "role",
            "cost_spice_eq",
            "effective_build_time_text",
            "units_per_hour_effective",
            "attack_effective",
            "shield_effective",
            "hull_effective",
            "speed_effective",
            "cargo",
            "fuel",
            "prerequisites",
        ]
    ].to_string(index=False)
)

### 6.4 Planetary Defenses (`d1`–`d11`) & Master Parameter Registry (`defenses_df`, `GAME_PARAMS_TABLES`)

Finally, this component builds **`defenses_df`** (`11` planetary turrets and shield domes indexed via `DEFENSE_SLUG_BY_KEY`, including unit resource costs, `cost_spice_eq`, `points_value`, *Weapons factory* build times, base & tech-boosted `hull` / `shield` / `attack`, `can_dig_in` status, cost-efficiency ratios, and `prerequisites`) and registers all **7 `pandas.DataFrame` parameter tables** inside **`GAME_PARAMS_TABLES`** for direct lookup by the Full Planner strategy simulator.

In [ ]:
_def_rows: list[dict[str, Any]] = []
for _dkey in sorted(DEFENSE_SLUG_BY_KEY.keys(), key=lambda k: int(k[1:])):
    _did = int(_dkey[1:])
    _slug = DEFENSE_SLUG_BY_KEY[_dkey]
    _dinfo = game_data.DEFENSES[_slug]
    _cost = dict(_dinfo.get("base_cost") or {})
    for _rk in RESOURCE_KEYS:
        _cost.setdefault(_rk, 0)
    _cost_sp = float(game_data.spice_equivalent(_cost))
    _pts_val = float(sum(int(_cost[k]) for k in RESOURCE_KEYS)) / float(POINTS_PER_RESOURCE_DIVISOR)
    _base_bt = float(_dinfo.get("base_build_time_s") or 0.0)
    _eff_bt = _base_bt / SHIPYARD_TIME_DIVISOR
    _units_hr = (3600.0 / _eff_bt) if _eff_bt > 0.0 else 0.0

    _st = _dinfo.get("stats") or {}
    _hull_b = int(_st.get("hull") or 0)
    _shld_b = int(_st.get("shield") or 0)
    _atk_b = int(_st.get("attack") or 0)
    _is_dome = _dkey in ("d10", "d11")
    _hull_eff = float(_hull_b) * _hul_mult
    _shld_eff = float(_shld_b) * _shd_mult
    _atk_eff = float(_atk_b) * _atk_mult

    _def_rows.append(
        {
            "defense_id": _did,
            "key": _dkey,
            "slug": _slug,
            "name": str(_dinfo["name"]),
            "role": "shield_dome" if _is_dome else "planetary_turret",
            "can_dig_in": not _is_dome,
            "pig_iron": int(_cost["pig_iron"]),
            "metal": int(_cost["metal"]),
            "kryptonite": int(_cost["kryptonite"]),
            "spice": int(_cost["spice"]),
            "cost_spice_eq": round(_cost_sp, 2),
            "points_value": round(_pts_val, 3),
            "base_build_time_s": round(_base_bt, 2),
            "effective_build_time_s": round(_eff_bt, 2),
            "effective_build_time_text": fmt_duration(_eff_bt),
            "units_per_hour_effective": round(_units_hr, 2),
            "hull_base": _hull_b,
            "shield_base": _shld_b,
            "attack_base": _atk_b,
            "hull_effective": round(_hull_eff, 1),
            "shield_effective": round(_shld_eff, 1),
            "attack_effective": round(_atk_eff, 1),
            "attack_per_spice_eq": round(_atk_eff / _cost_sp, 4) if _cost_sp > 0.0 else 0.0,
            "hull_plus_shield_per_spice_eq": round((_hull_eff + _shld_eff) / _cost_sp, 2) if _cost_sp > 0.0 else 0.0,
            "prerequisites": format_prerequisites(_dinfo.get("requirements") or []),
        }
    )

defenses_df = pd.DataFrame(_def_rows)

GAME_PARAMS_TABLES: dict[str, pd.DataFrame] = {
    "merchant_rates": merchant_rates_df,
    "buildings_catalog": buildings_catalog_df,
    "buildings_levels": buildings_levels_df,
    "technologies_catalog": technologies_catalog_df,
    "technologies_levels": technologies_levels_df,
    "ships": ships_df,
    "defenses": defenses_df,
}

game_params_summary_df = pd.DataFrame(
    [
        {
            "table_name": _tname,
            "dataframe_var": f"{_tname}_df",
            "rows": int(_df.shape[0]),
            "columns": int(_df.shape[1]),
            "sample_columns": ", ".join(list(_df.columns[:6])),
        }
        for _tname, _df in GAME_PARAMS_TABLES.items()
    ]
)

print(f"Built defenses_df ({defenses_df.shape[0]} rows x {defenses_df.shape[1]} cols)")
print("\n[defenses_df — All 11 Planetary Defenses]")
print(
    defenses_df[
        [
            "key",
            "name",
            "role",
            "can_dig_in",
            "cost_spice_eq",
            "effective_build_time_text",
            "attack_effective",
            "shield_effective",
            "hull_effective",
            "attack_per_spice_eq",
            "prerequisites",
        ]
    ].to_string(index=False)
)
print("\n[GAME_PARAMS_TABLES — Registered Pandas DataFrames]")
print(game_params_summary_df.to_string(index=False))

## 7. In-Memory Technology & Prerequisite Requirement Graph (`TECH_GRAPH`)

To evaluate legal build/research/shipyard transitions in $O(1)$–$O(V+E)$ memory operations during thousands of strategy rollouts without touching SQLite or re-parsing strings, this section constructs an **in-memory Directed Acyclic Graph (`TechnologyRequirementGraph`)** spanning all **62 SpaceInvasion Classic entities** (`21` buildings `b1`–`b21`, `15` technologies `r1`–`r15`, `15` spaceships `s1`–`s16`, and `11` defenses `d1`–`d11`) and all **101 weighted prerequisite edges** (`source_key @ required_level ──▶ target_key`).

### 7.1 Graph Construction, Kahn Topological Sort & Adjacency Tables (`tech_graph_nodes_df`, `tech_graph_edges_df`)

`TechnologyRequirementGraph` indexes in memory:
- **`prereqs_of[key]`** (incoming requirement edges `(source_key, required_level)`) and **`unlocks_of[key]`** (outgoing gated dependents `(target_key, required_level)`).
- **Kahn topological ordering (`topological_order`) & depth (`topological_layer`)**: Verifies strict DAG acyclicity (`len(topological_order) == 62`) and assigns every entity its exact dependency tier (`Layer 0` root mines/power/Dev Center $\rightarrow$ `Layer 1` Research Lab $\rightarrow$ $\dots$ $\rightarrow$ endgame *Imperial starbase* & *Quantum Array*).
- **`tech_graph_nodes_df` (`62` rows)** and **`tech_graph_edges_df` (`101` rows)**: `pandas.DataFrame` views over the in-memory graph annotated against the active colony's current `buildings` and `researches`.

In [ ]:
class TechnologyRequirementGraph:
    """In-memory Directed Acyclic Graph (DAG) of SpaceInvasion Classic tech-tree requirements."""

    def __init__(self) -> None:
        self.nodes: dict[str, dict[str, Any]] = {}
        self.slug_to_key: dict[tuple[str, str], str] = {}
        self._raw_reqs: dict[str, list[dict[str, Any]]] = {}
        self._mono_times_by_key: dict[str, dict[int, float]] = {}

        for bid in sorted(game_data.BUILDING_BY_ID.keys()):
            info = game_data.BUILDING_BY_ID[bid]
            bkey = f"b{bid}"
            max_lvl = int(info.get("max_table_level") or len(info.get("levels") or {}))
            self.nodes[bkey] = {
                "key": bkey,
                "category": "buildings",
                "entity_id": bid,
                "slug": str(info["slug"]),
                "name": str(info["name"]),
                "max_level": max_lvl,
            }
            self.slug_to_key[("buildings", str(info["slug"]))] = bkey
            self._raw_reqs[bkey] = list(info.get("requirements") or [])
            self._mono_times_by_key[bkey] = build_monotonic_base_times(info.get("levels") or {}, max_lvl)

        for rnum, rkey, rinfo in sorted(
            (
                (int(str(v["id"])[1:]), str(v["id"]), v)
                for v in game_data.RESEARCHES.values()
                if isinstance(v.get("id"), str) and str(v["id"]).startswith("r") and str(v["id"])[1:].isdigit()
            ),
            key=lambda item: item[0],
        ):
            max_lvl = int(rinfo.get("max_table_level") or len(rinfo.get("levels") or {}))
            self.nodes[rkey] = {
                "key": rkey,
                "category": "research",
                "entity_id": rnum,
                "slug": str(rinfo["slug"]),
                "name": str(rinfo["name"]),
                "max_level": max_lvl,
            }
            self.slug_to_key[("research", str(rinfo["slug"]))] = rkey
            self._raw_reqs[rkey] = list(rinfo.get("requirements") or [])
            self._mono_times_by_key[rkey] = build_monotonic_base_times(rinfo.get("levels") or {}, max_lvl)

        for skey in sorted(SHIP_SLUG_BY_KEY.keys(), key=lambda k: int(k[1:])):
            slug = SHIP_SLUG_BY_KEY[skey]
            sinfo = game_data.SHIPS[slug]
            self.nodes[skey] = {
                "key": skey,
                "category": "ships",
                "entity_id": int(skey[1:]),
                "slug": slug,
                "name": str(sinfo["name"]),
                "max_level": 0,
            }
            self.slug_to_key[("ships", slug)] = skey
            self._raw_reqs[skey] = list(sinfo.get("requirements") or [])

        for dkey in sorted(DEFENSE_SLUG_BY_KEY.keys(), key=lambda k: int(k[1:])):
            slug = DEFENSE_SLUG_BY_KEY[dkey]
            dinfo = game_data.DEFENSES[slug]
            self.nodes[dkey] = {
                "key": dkey,
                "category": "defense",
                "entity_id": int(dkey[1:]),
                "slug": slug,
                "name": str(dinfo["name"]),
                "max_level": 0,
            }
            self.slug_to_key[("defense", slug)] = dkey
            self._raw_reqs[dkey] = list(dinfo.get("requirements") or [])

        prereqs_mut: dict[str, list[tuple[str, int]]] = {k: [] for k in self.nodes}
        unlocks_mut: dict[str, list[tuple[str, int]]] = {k: [] for k in self.nodes}
        for target_key, req_list in self._raw_reqs.items():
            for req in req_list:
                src_key = self.slug_to_key.get((str(req["category"]), str(req["slug"])))
                if src_key is None:
                    raise KeyError(f"Unresolved tech-tree requirement {req!r} on {target_key}")
                req_lvl = int(req["level"])
                prereqs_mut[target_key].append((src_key, req_lvl))
                unlocks_mut[src_key].append((target_key, req_lvl))

        self.prereqs_of: dict[str, tuple[tuple[str, int], ...]] = {
            k: tuple(v) for k, v in prereqs_mut.items()
        }
        self.unlocks_of: dict[str, tuple[tuple[str, int], ...]] = {
            k: tuple(v) for k, v in unlocks_mut.items()
        }

        # Kahn's algorithm for strict DAG validation & topological layering
        in_deg: dict[str, int] = {k: len(self.prereqs_of[k]) for k in self.nodes}
        layer: dict[str, int] = {k: 0 for k in self.nodes}
        queue: deque[str] = deque(k for k, deg in in_deg.items() if deg == 0)
        topo: list[str] = []
        while queue:
            u = queue.popleft()
            topo.append(u)
            for v, _lvl in self.unlocks_of[u]:
                if layer[u] + 1 > layer[v]:
                    layer[v] = layer[u] + 1
                in_deg[v] -= 1
                if in_deg[v] == 0:
                    queue.append(v)

        if len(topo) != len(self.nodes):
            cyclic = sorted(set(self.nodes) - set(topo))
            raise ValueError(f"Cycle detected in technology requirement graph: {cyclic}")

        self.topological_order: tuple[str, ...] = tuple(topo)
        self.topological_layer: dict[str, int] = layer
        self._topo_rank: dict[str, int] = {k: i for i, k in enumerate(self.topological_order)}

        # Precompute transitive ancestors & descendants in O(V * (V + E)) once
        anc: dict[str, set[str]] = {k: set() for k in self.nodes}
        for u in self.topological_order:
            for p_key, _ in self.prereqs_of[u]:
                anc[u].add(p_key)
                anc[u].update(anc[p_key])
        desc: dict[str, set[str]] = {k: set() for k in self.nodes}
        for u in reversed(self.topological_order):
            for c_key, _ in self.unlocks_of[u]:
                desc[u].add(c_key)
                desc[u].update(desc[c_key])

        self.ancestors_of: dict[str, frozenset[str]] = {k: frozenset(v) for k, v in anc.items()}
        self.descendants_of: dict[str, frozenset[str]] = {k: frozenset(v) for k, v in desc.items()}

    def current_level(
        self,
        key: str,
        buildings: dict[str, int],
        researches: dict[str, int],
    ) -> int:
        node = self.nodes[key]
        if node["category"] == "buildings":
            return max(0, int(buildings.get(key, 0) or 0))
        if node["category"] == "research":
            return max(0, int(researches.get(key, 0) or 0))
        return 0

    def is_unlocked(
        self,
        target_key: str,
        buildings: dict[str, int],
        researches: dict[str, int],
    ) -> bool:
        """O(in_degree) in-memory prerequisite satisfaction check."""
        for src_key, req_lvl in self.prereqs_of[target_key]:
            if self.current_level(src_key, buildings, researches) < req_lvl:
                return False
        return True

    def direct_missing(
        self,
        target_key: str,
        buildings: dict[str, int],
        researches: dict[str, int],
    ) -> list[dict[str, Any]]:
        missing: list[dict[str, Any]] = []
        for src_key, req_lvl in self.prereqs_of[target_key]:
            cur = self.current_level(src_key, buildings, researches)
            if cur < req_lvl:
                src_node = self.nodes[src_key]
                missing.append(
                    {
                        "key": src_key,
                        "name": src_node["name"],
                        "category": src_node["category"],
                        "required_level": req_lvl,
                        "current_level": cur,
                        "missing_levels": req_lvl - cur,
                    }
                )
        return missing

    def transitive_requirements(
        self,
        target_key: str,
        *,
        include_target_level: int = 0,
    ) -> dict[str, int]:
        """Resolve minimum required `{node_key: min_level}` across all diamond ancestor paths."""
        if target_key not in self.nodes:
            raise KeyError(f"Unknown node key: {target_key!r}")
        req_levels: dict[str, int] = {}
        if include_target_level > 0 and self.nodes[target_key]["category"] in ("buildings", "research"):
            req_levels[target_key] = int(include_target_level)
        for src_key, req_lvl in self.prereqs_of[target_key]:
            req_levels[src_key] = max(req_levels.get(src_key, 0), int(req_lvl))

        # Walk backward in reverse topological order so diamond convergences take exact max(level)
        for node_key in reversed(self.topological_order):
            if node_key not in req_levels:
                continue
            for p_key, p_lvl in self.prereqs_of[node_key]:
                if p_lvl > req_levels.get(p_key, 0):
                    req_levels[p_key] = int(p_lvl)

        return {
            k: req_levels[k]
            for k in sorted(req_levels.keys(), key=lambda item: self._topo_rank[item])
        }

    def _single_level_cost_and_base_time(self, key: str, level: int) -> tuple[dict[str, int], float]:
        node = self.nodes[key]
        if node["category"] == "buildings":
            cost = game_data.building_cost(int(node["entity_id"]), level)
            bt = float(self._mono_times_by_key[key].get(level, 0.0))
            return cost, bt
        if node["category"] == "research":
            rinfo = game_data.RESEARCHES[node["slug"]]
            row = (rinfo.get("levels") or {}).get(level) or {}
            cost = dict(row.get("cost") or {})
            for rk in RESOURCE_KEYS:
                cost.setdefault(rk, 0)
            bt = float(self._mono_times_by_key[key].get(level, 0.0))
            return cost, bt
        return {rk: 0 for rk in RESOURCE_KEYS}, 0.0

    def plan_unlock_sequence(
        self,
        target_key: str,
        buildings: dict[str, int],
        researches: dict[str, int],
        *,
        speedup_factor: float = 5.0,
        include_target_level: int = 0,
    ) -> pd.DataFrame:
        """Build topologically ordered upgrade steps from `(buildings, researches)` to unlock `target_key`."""
        req_map = self.transitive_requirements(
            target_key,
            include_target_level=include_target_level,
        )
        sim_b = {str(k): max(0, int(v or 0)) for k, v in buildings.items()}
        sim_r = {str(k): max(0, int(v or 0)) for k, v in researches.items()}
        spd = max(0.1, float(speedup_factor))

        rows: list[dict[str, Any]] = []
        cum_sp = 0.0
        cum_time_s = 0.0
        step_idx = 0

        for nkey, req_lvl in req_map.items():
            node = self.nodes[nkey]
            cur_lvl = self.current_level(nkey, sim_b, sim_r)
            if cur_lvl >= req_lvl:
                continue
            step_cost = {rk: 0 for rk in RESOURCE_KEYS}
            step_time_s = 0.0
            for nxt_lvl in range(cur_lvl + 1, req_lvl + 1):
                c_dict, bt_s = self._single_level_cost_and_base_time(nkey, nxt_lvl)
                for rk in RESOURCE_KEYS:
                    step_cost[rk] += int(c_dict.get(rk, 0))
                if node["category"] == "buildings":
                    div = max(
                        1.0,
                        float(sim_b.get("b7", 0) + 1)
                        * float(2 ** min(sim_b.get("b17", 0), 40))
                        * spd,
                    )
                    step_time_s += bt_s / div
                    sim_b[nkey] = nxt_lvl
                else:
                    div = max(1.0, float(sim_b.get("b8", 0) + 1) * spd)
                    step_time_s += bt_s / div
                    sim_r[nkey] = nxt_lvl

            step_idx += 1
            step_sp = float(game_data.spice_equivalent(step_cost))
            cum_sp += step_sp
            cum_time_s += step_time_s
            rows.append(
                {
                    "step": step_idx,
                    "key": nkey,
                    "name": node["name"],
                    "category": node["category"],
                    "topological_layer": self.topological_layer[nkey],
                    "from_level": cur_lvl,
                    "to_level": req_lvl,
                    "levels_added": req_lvl - cur_lvl,
                    "pig_iron": step_cost["pig_iron"],
                    "metal": step_cost["metal"],
                    "kryptonite": step_cost["kryptonite"],
                    "spice": step_cost["spice"],
                    "step_cost_spice_eq": round(step_sp, 2),
                    "step_effective_time_s": round(step_time_s, 2),
                    "step_effective_time_text": fmt_duration(step_time_s),
                    "cum_cost_spice_eq": round(cum_sp, 2),
                    "cum_effective_time_s": round(cum_time_s, 2),
                    "cum_effective_time_text": fmt_duration(cum_time_s),
                }
            )

        return pd.DataFrame(
            rows,
            columns=[
                "step",
                "key",
                "name",
                "category",
                "topological_layer",
                "from_level",
                "to_level",
                "levels_added",
                "pig_iron",
                "metal",
                "kryptonite",
                "spice",
                "step_cost_spice_eq",
                "step_effective_time_s",
                "step_effective_time_text",
                "cum_cost_spice_eq",
                "cum_effective_time_s",
                "cum_effective_time_text",
            ],
        )


TECH_GRAPH = TechnologyRequirementGraph()

_active_bld = telemetry_bundle.get("colony_buildings") or {}
_active_res = telemetry_bundle.get("colony_researches") or {}

tech_graph_nodes_df = pd.DataFrame(
    [
        {
            "key": _k,
            "category": TECH_GRAPH.nodes[_k]["category"],
            "slug": TECH_GRAPH.nodes[_k]["slug"],
            "name": TECH_GRAPH.nodes[_k]["name"],
            "topological_layer": TECH_GRAPH.topological_layer[_k],
            "in_degree": len(TECH_GRAPH.prereqs_of[_k]),
            "out_degree": len(TECH_GRAPH.unlocks_of[_k]),
            "transitive_prereq_count": len(TECH_GRAPH.ancestors_of[_k]),
            "transitive_unlock_count": len(TECH_GRAPH.descendants_of[_k]),
            "is_root_node": len(TECH_GRAPH.prereqs_of[_k]) == 0,
            "current_level": TECH_GRAPH.current_level(_k, _active_bld, _active_res),
            "unlocked_now": TECH_GRAPH.is_unlocked(_k, _active_bld, _active_res),
        }
        for _k in TECH_GRAPH.topological_order
    ]
)

tech_graph_edges_df = pd.DataFrame(
    [
        {
            "source_key": _src,
            "source_name": TECH_GRAPH.nodes[_src]["name"],
            "source_category": TECH_GRAPH.nodes[_src]["category"],
            "required_level": _req_lvl,
            "target_key": _tgt,
            "target_name": TECH_GRAPH.nodes[_tgt]["name"],
            "target_category": TECH_GRAPH.nodes[_tgt]["category"],
            "target_layer": TECH_GRAPH.topological_layer[_tgt],
            "source_current_level": TECH_GRAPH.current_level(_src, _active_bld, _active_res),
            "satisfied_now": TECH_GRAPH.current_level(_src, _active_bld, _active_res) >= _req_lvl,
            "levels_missing": max(0, _req_lvl - TECH_GRAPH.current_level(_src, _active_bld, _active_res)),
        }
        for _tgt in TECH_GRAPH.topological_order
        for _src, _req_lvl in TECH_GRAPH.prereqs_of[_tgt]
    ]
)

print(
    f"Constructed in-memory TECH_GRAPH: {len(TECH_GRAPH.nodes)} nodes, "
    f"{len(tech_graph_edges_df)} directed edges, "
    f"max topological depth = {max(TECH_GRAPH.topological_layer.values())} (strict DAG verified)"
)
print("\n[Top Hub Nodes by Transitive Unlocks (`tech_graph_nodes_df`)]")
print(
    tech_graph_nodes_df.sort_values(
        ["transitive_unlock_count", "out_degree"], ascending=[False, False]
    )
    .head(12)[
        [
            "key",
            "name",
            "category",
            "topological_layer",
            "in_degree",
            "out_degree",
            "transitive_unlock_count",
            "current_level",
            "unlocked_now",
        ]
    ]
    .to_string(index=False)
)

### 7.2 Transitive Prerequisite Resolver & Dynamic Unlock Path Planner (`unlock_targets_summary_df`, `phoenix_unlock_plan_df`)

Using the in-memory `TECH_GRAPH`, this component resolves **diamond multi-hop prerequisite chains** (`TECH_GRAPH.transitive_requirements`) and computes **topologically ordered unlock plans** (`TECH_GRAPH.plan_unlock_sequence`) for every building, technology, spaceship, and defense:

- **Diamond Dependency Convergence**: When multiple prerequisite branches converge on the same ancestor (e.g., `s7` *Phoenix* requires `b9 L7` $\rightarrow$ `b7 L3` and `r10 L5` $\rightarrow$ `b8 L5` $\rightarrow$ `b7 L2`), reverse-topological propagation takes the exact `max(required_level)` across all paths without double-counting costs or levels.
- **Dynamic Accelerator Accounting**: As `plan_unlock_sequence` steps forward in topological order, upgrades to `b7` (*Development center*), `b8` (*Research lab*), and `b17` (*Microsystem accelerator*) immediately accelerate subsequent building and research steps in the same unlock chain.
- **`unlock_targets_summary_df` (`62` rows)** and **`phoenix_unlock_plan_df`**: Summarize remaining unlock Spice-equivalent cost, cumulative effective time, and ordered upgrade steps from the active colony state, and register the graph tables inside `GAME_PARAMS_TABLES`.

In [ ]:
_spd_now = float(GAME_PARAM_CONTEXT["speedup_factor"])
_unlock_summary_rows: list[dict[str, Any]] = []

for _nkey in TECH_GRAPH.topological_order:
    _node = TECH_GRAPH.nodes[_nkey]
    _plan_df = TECH_GRAPH.plan_unlock_sequence(
        _nkey,
        _active_bld,
        _active_res,
        speedup_factor=_spd_now,
    )
    _tot_sp = float(_plan_df["step_cost_spice_eq"].sum()) if not _plan_df.empty else 0.0
    _tot_t = float(_plan_df["step_effective_time_s"].sum()) if not _plan_df.empty else 0.0
    _tot_lvls = int(_plan_df["levels_added"].sum()) if not _plan_df.empty else 0
    _chain_str = (
        " -> ".join(
            f"{row['key']} L{int(row['from_level'])}->{int(row['to_level'])}"
            for _, row in _plan_df.iterrows()
        )
        if not _plan_df.empty
        else "Unlocked"
    )
    _unlock_summary_rows.append(
        {
            "key": _nkey,
            "name": _node["name"],
            "category": _node["category"],
            "topological_layer": TECH_GRAPH.topological_layer[_nkey],
            "unlocked_now": bool(_plan_df.empty),
            "missing_upgrade_nodes": int(len(_plan_df)),
            "total_levels_to_add": _tot_lvls,
            "unlock_cost_spice_eq": round(_tot_sp, 2),
            "unlock_effective_time_s": round(_tot_t, 2),
            "unlock_effective_time_text": fmt_duration(_tot_t) if _tot_t > 0.0 else "Ready",
            "ordered_prerequisite_chain": _chain_str,
        }
    )

unlock_targets_summary_df = pd.DataFrame(_unlock_summary_rows)

phoenix_unlock_plan_df = TECH_GRAPH.plan_unlock_sequence(
    "s7",
    _active_bld,
    _active_res,
    speedup_factor=_spd_now,
)
cold_phoenix_unlock_plan_df = TECH_GRAPH.plan_unlock_sequence(
    "s7",
    {},
    {},
    speedup_factor=_spd_now,
)

GAME_PARAMS_TABLES["tech_graph_nodes"] = tech_graph_nodes_df
GAME_PARAMS_TABLES["tech_graph_edges"] = tech_graph_edges_df
GAME_PARAMS_TABLES["unlock_targets_summary"] = unlock_targets_summary_df

print(
    f"Evaluated unlock paths across {len(unlock_targets_summary_df)} nodes: "
    f"{int(unlock_targets_summary_df['unlocked_now'].sum())} unlocked now, "
    f"{int((~unlock_targets_summary_df['unlocked_now']).sum())} locked"
)
print("\n[Strategic Locked Milestones — Ordered Unlock Cost & Time (`unlock_targets_summary_df`)]")
_locked_df = unlock_targets_summary_df[~unlock_targets_summary_df["unlocked_now"]].sort_values(
    ["unlock_cost_spice_eq", "topological_layer"]
)
if not _locked_df.empty:
    print(
        _locked_df.head(15)[
            [
                "key",
                "name",
                "category",
                "topological_layer",
                "total_levels_to_add",
                "unlock_cost_spice_eq",
                "unlock_effective_time_text",
                "ordered_prerequisite_chain",
            ]
        ].to_string(index=False)
    )

print("\n[Step-by-Step Topological Unlock Plan for `s7` Phoenix from Active Colony (`phoenix_unlock_plan_df`)]")
if not phoenix_unlock_plan_df.empty:
    print(
        phoenix_unlock_plan_df[
            [
                "step",
                "key",
                "name",
                "from_level",
                "to_level",
                "step_cost_spice_eq",
                "step_effective_time_text",
                "cum_cost_spice_eq",
                "cum_effective_time_text",
            ]
        ].to_string(index=False)
    )
else:
    print("s7 Phoenix is already unlocked on the active colony!")

## 8. Empire Planet Sizes, Live Planet States & Empty 300-Field Colony Slots

To simulate full-empire growth from the current account state through full colonization, this section loads the **exact size (`fields_used` / `fields_max`) and state** (resources, storage caps, energy balance, buildings `b1`–`b21`, ships, and defenses) of every currently owned planet from `state/spaceinvasion.db`, and automatically pads any remaining uncolonized empire slots up to **`MAX_EMPIRE_PLANETS`** (`11` in SpaceInvasion Classic RETRO) with **empty `300`-field colony slots**.

### 8.1 Load Live Planet Sizes & States from SQLite (`live_planet_states`, `live_planets_df`)

Reads the latest planet states safely from an isolated temp copy of `state/spaceinvasion.db` (preferring `meta['latest_game_state']` for exact live warehouse capacities and falling back to `planet_snapshots` grouped by `planet_id`), producing typed `PlanetState` objects and a summary `pandas.DataFrame` (`live_planets_df`).

In [ ]:
MAX_EMPIRE_PLANETS: int = 11
EMPTY_COLONY_SIZE_FIELDS: int = 300
EMPTY_COLONY_DEFAULT_TEMP_C: int = 0

SQL_LATEST_PLANETS_PER_ID = """
SELECT
    p.planet_id,
    p.coords,
    p.name,
    p.planet_type,
    p.temperature,
    p.fields_used,
    p.fields_max,
    p.pig_iron,
    p.metal,
    p.kryptonite,
    p.spice,
    p.energy_produced,
    p.energy_used,
    p.buildings_json,
    p.ships_json,
    p.defense_json,
    p.economy_rates_json
FROM planet_snapshots p
INNER JOIN (
    SELECT planet_id, MAX(id) AS max_id
    FROM planet_snapshots
    GROUP BY planet_id
) latest ON latest.max_id = p.id
ORDER BY p.planet_id ASC
"""


def _warehouse_cap_from_buildings(buildings: dict[str, int], b_id: int) -> int:
    lvl = max(0, min(40, int(buildings.get(f"b{b_id}", 0) or 0)))
    return int(100_000 * (2**lvl))


def load_live_planet_states_readonly(db_path: Path) -> list[PlanetState]:
    """Load all owned planets' sizes and typed `PlanetState` objects from a temp copy of SQLite."""
    if not db_path.is_file() or db_path.stat().st_size < 100:
        return []

    planets_out: list[PlanetState] = []
    with tempfile.TemporaryDirectory(prefix="si_fp_planets_") as tmp_dir:
        tmp_root = Path(tmp_dir)
        tmp_db = tmp_root / db_path.name
        shutil.copy2(db_path, tmp_db)
        tmp_db.chmod(0o600)
        for suffix in ("-wal", "-shm", "-journal"):
            sidecar = Path(f"{db_path}{suffix}")
            if sidecar.is_file():
                tmp_sidecar = Path(f"{tmp_db}{suffix}")
                shutil.copy2(sidecar, tmp_sidecar)
                tmp_sidecar.chmod(0o600)

        ro_db = GameDatabase(
            tmp_db,
            attack_history_path=tmp_root / "ro_mirror.json",
            backup_on_upgrade=False,
        )
        try:
            conn = ro_db._conn
            meta_row = conn.execute(SQL_LATEST_GAME_STATE_META).fetchone()
            gs_planets = (
                (_safe_json_dict(meta_row[0]).get("planets") or [])
                if (meta_row and meta_row[0])
                else []
            )
            if isinstance(gs_planets, list) and gs_planets:
                for raw_p in gs_planets:
                    if not isinstance(raw_p, dict):
                        continue
                    bld_map = {
                        str(k): max(0, int(v or 0))
                        for k, v in _safe_json_dict(raw_p.get("buildings")).items()
                        if isinstance(v, (int, float))
                    }
                    shp_map = {
                        str(k): max(0, int(v or 0))
                        for k, v in _safe_json_dict(raw_p.get("ships")).items()
                        if isinstance(v, (int, float))
                    }
                    def_map = {
                        str(k): max(0, int(v or 0))
                        for k, v in _safe_json_dict(raw_p.get("defense")).items()
                        if isinstance(v, (int, float))
                    }
                    econ_map = {
                        str(k): int(v or 0)
                        for k, v in _safe_json_dict(raw_p.get("economy_rates")).items()
                        if isinstance(v, (int, float))
                    }
                    res_d = _safe_json_dict(raw_p.get("resources"))
                    sto_d = _safe_json_dict(raw_p.get("storage"))
                    eng_d = _safe_json_dict(raw_p.get("energy"))
                    f_used = int(raw_p.get("fields_used") or sum(bld_map.values()))
                    f_max = int(raw_p.get("fields_max") or max(230, f_used))
                    planets_out.append(
                        PlanetState(
                            planet_id=int(raw_p.get("planet_id") or 0),
                            name=str(raw_p.get("name") or "Colony"),
                            coords=str(raw_p.get("coords") or "00:000:00"),
                            planet_type=str(raw_p.get("planet_type") or "normal"),
                            temperature=int(raw_p.get("temperature") or 0),
                            fields_used=f_used,
                            fields_max=f_max,
                            resources=Resources(
                                pig_iron=int(res_d.get("pig_iron") or 0),
                                metal=int(res_d.get("metal") or 0),
                                kryptonite=int(res_d.get("kryptonite") or 0),
                                spice=int(res_d.get("spice") or 0),
                            ),
                            storage=Resources(
                                pig_iron=int(sto_d.get("pig_iron") or _warehouse_cap_from_buildings(bld_map, 11)),
                                metal=int(sto_d.get("metal") or _warehouse_cap_from_buildings(bld_map, 12)),
                                kryptonite=int(sto_d.get("kryptonite") or _warehouse_cap_from_buildings(bld_map, 13)),
                                spice=int(sto_d.get("spice") or _warehouse_cap_from_buildings(bld_map, 14)),
                            ),
                            energy=EnergyBalance(
                                produced=int(eng_d.get("produced") or 0),
                                used=int(eng_d.get("used") or 0),
                            ),
                            buildings=bld_map,
                            ships=shp_map,
                            defense=def_map,
                            economy_rates=econ_map,
                        )
                    )
            else:
                for row in conn.execute(SQL_LATEST_PLANETS_PER_ID).fetchall():
                    bld_map = {
                        str(k): max(0, int(v or 0))
                        for k, v in _safe_json_dict(row["buildings_json"]).items()
                        if isinstance(v, (int, float))
                    }
                    shp_map = {
                        str(k): max(0, int(v or 0))
                        for k, v in _safe_json_dict(row["ships_json"]).items()
                        if isinstance(v, (int, float))
                    }
                    def_map = {
                        str(k): max(0, int(v or 0))
                        for k, v in _safe_json_dict(row["defense_json"]).items()
                        if isinstance(v, (int, float))
                    }
                    econ_map = {
                        str(k): int(v or 0)
                        for k, v in _safe_json_dict(row["economy_rates_json"]).items()
                        if isinstance(v, (int, float))
                    }
                    f_used = int(row["fields_used"] or sum(bld_map.values()))
                    f_max = int(row["fields_max"] or max(230, f_used))
                    planets_out.append(
                        PlanetState(
                            planet_id=int(row["planet_id"]),
                            name=str(row["name"] or "Colony"),
                            coords=str(row["coords"] or "00:000:00"),
                            planet_type=str(row["planet_type"] or "normal"),
                            temperature=int(row["temperature"] or 0),
                            fields_used=f_used,
                            fields_max=f_max,
                            resources=Resources(
                                pig_iron=int(row["pig_iron"] or 0),
                                metal=int(row["metal"] or 0),
                                kryptonite=int(row["kryptonite"] or 0),
                                spice=int(row["spice"] or 0),
                            ),
                            storage=Resources(
                                pig_iron=_warehouse_cap_from_buildings(bld_map, 11),
                                metal=_warehouse_cap_from_buildings(bld_map, 12),
                                kryptonite=_warehouse_cap_from_buildings(bld_map, 13),
                                spice=_warehouse_cap_from_buildings(bld_map, 14),
                            ),
                            energy=EnergyBalance(
                                produced=int(row["energy_produced"] or 0),
                                used=int(row["energy_used"] or 0),
                            ),
                            buildings=bld_map,
                            ships=shp_map,
                            defense=def_map,
                            economy_rates=econ_map,
                        )
                    )
        finally:
            ro_db.close()

    return planets_out


def _planet_state_to_row(slot_idx: int, planet: PlanetState, *, is_empty_slot: bool) -> dict[str, Any]:
    res_sp = float(
        game_data.spice_equivalent(
            {
                "pig_iron": planet.resources.pig_iron,
                "metal": planet.resources.metal,
                "kryptonite": planet.resources.kryptonite,
                "spice": planet.resources.spice,
            }
        )
    )
    f_max = int(planet.fields_max)
    f_used = int(planet.fields_used)
    f_rem = int(planet.fields_remaining)
    util_pct = (100.0 * float(f_used) / float(f_max)) if f_max > 0 else 0.0
    return {
        "slot_index": slot_idx,
        "planet_id": int(planet.planet_id),
        "status": "empty_colony_slot" if is_empty_slot else "colonized",
        "is_empty_slot": bool(is_empty_slot),
        "name": planet.name,
        "coords": planet.coords,
        "planet_type": planet.planet_type,
        "temperature_c": int(planet.temperature),
        "fields_used": f_used,
        "fields_max": f_max,
        "fields_remaining": f_rem,
        "field_utilization_pct": round(util_pct, 2),
        "pig_iron": int(planet.resources.pig_iron),
        "metal": int(planet.resources.metal),
        "kryptonite": int(planet.resources.kryptonite),
        "spice": int(planet.resources.spice),
        "resources_spice_eq": round(res_sp, 2),
        "storage_pig_iron": int(planet.storage.pig_iron),
        "storage_metal": int(planet.storage.metal),
        "storage_kryptonite": int(planet.storage.kryptonite),
        "storage_spice": int(planet.storage.spice),
        "energy_produced": int(planet.energy.produced),
        "energy_used": int(planet.energy.used),
        "energy_net": int(planet.energy.produced) - int(planet.energy.used),
        "total_building_levels": int(sum(planet.buildings.values())),
        "total_ships": int(sum(planet.ships.values())),
        "total_defenses": int(sum(planet.defense.values())),
    }


live_planet_states: list[PlanetState] = load_live_planet_states_readonly(LIVE_DB_PATH)
live_planets_df = pd.DataFrame(
    [
        _planet_state_to_row(idx + 1, p, is_empty_slot=False)
        for idx, p in enumerate(live_planet_states)
    ]
)

print(f"Loaded {len(live_planet_states)} colonized planet(s) from SQLite (MAX_EMPIRE_PLANETS={MAX_EMPIRE_PLANETS})")
if not live_planets_df.empty:
    print("\n[live_planets_df — Owned Planet Sizes & Resource/Energy State]")
    print(
        live_planets_df[
            [
                "planet_id",
                "name",
                "coords",
                "planet_type",
                "temperature_c",
                "fields_used",
                "fields_max",
                "fields_remaining",
                "field_utilization_pct",
                "resources_spice_eq",
                "energy_net",
                "total_building_levels",
                "total_ships",
            ]
        ].to_string(index=False)
    )

### 8.2 Pad Empire Roster to `MAX_EMPIRE_PLANETS` with Empty `300`-Field Colony Spaces (`empire_planets_df`, `empire_planet_buildings_df`)

Whenever the player owns fewer than `MAX_EMPIRE_PLANETS` (`11`) normal planets, this component creates the remaining **`MAX_EMPIRE_PLANETS - owned_normal_count` empty colony slots**, each initialized with:
- **`fields_max = 300`** (`EMPTY_COLONY_SIZE_FIELDS = 300`, `fields_used = 0`, `fields_remaining = 300`).
- **Clean `PlanetState` baseline**: `planet_id = -1..-K`, `coords = "EMPTY:01".."EMPTY:KK"`, `0` resources, base `100,000` L0 warehouse capacities, `0` energy, and `0` building/ship/defense levels.
- **Vectorized Matrices (`empire_planets_df`, `empire_planet_buildings_df`, `planet_sizes_arr`)**: Ready for full-empire colonization and multi-planet build-order simulation.

In [ ]:
def build_padded_empire_planets(
    live_planets: list[PlanetState],
    *,
    max_planets: int = MAX_EMPIRE_PLANETS,
    empty_planet_size: int = EMPTY_COLONY_SIZE_FIELDS,
    empty_temp_c: int = EMPTY_COLONY_DEFAULT_TEMP_C,
) -> tuple[list[PlanetState], pd.DataFrame, pd.DataFrame]:
    """Combine colonized `PlanetState`s with empty `empty_planet_size`-field colony slots up to `max_planets`."""
    all_states: list[PlanetState] = []
    rows: list[dict[str, Any]] = []
    building_rows: list[dict[str, Any]] = []
    all_bkeys = [f"b{bid}" for bid in sorted(game_data.BUILDING_BY_ID.keys())]

    normal_owned_count = 0
    for idx, p in enumerate(live_planets, start=1):
        all_states.append(p)
        if p.planet_type == "normal":
            normal_owned_count += 1
        rows.append(_planet_state_to_row(idx, p, is_empty_slot=False))
        b_row: dict[str, Any] = {
            "slot_index": idx,
            "planet_id": int(p.planet_id),
            "coords": p.coords,
            "status": "colonized",
            "fields_used": int(p.fields_used),
            "fields_max": int(p.fields_max),
        }
        for bk in all_bkeys:
            b_row[bk] = max(0, int(p.buildings.get(bk, 0) or 0))
        building_rows.append(b_row)

    missing_slots = max(0, int(max_planets) - normal_owned_count)
    for empty_idx in range(1, missing_slots + 1):
        slot_num = len(all_states) + 1
        empty_planet = PlanetState(
            planet_id=-empty_idx,
            name=f"Empty Colony Slot {empty_idx}",
            coords=f"EMPTY:{empty_idx:02d}",
            planet_type="normal",
            temperature=int(empty_temp_c),
            fields_used=0,
            fields_max=int(empty_planet_size),
            resources=Resources(0, 0, 0, 0),
            storage=Resources(100_000, 100_000, 100_000, 100_000),
            energy=EnergyBalance(0, 0),
            buildings={bk: 0 for bk in all_bkeys},
            ships={},
            defense={},
            economy_rates={},
        )
        all_states.append(empty_planet)
        rows.append(_planet_state_to_row(slot_num, empty_planet, is_empty_slot=True))
        b_row = {
            "slot_index": slot_num,
            "planet_id": int(empty_planet.planet_id),
            "coords": empty_planet.coords,
            "status": "empty_colony_slot",
            "fields_used": 0,
            "fields_max": int(empty_planet_size),
        }
        for bk in all_bkeys:
            b_row[bk] = 0
        building_rows.append(b_row)

    planets_df = pd.DataFrame(rows)
    buildings_df = pd.DataFrame(building_rows)
    return all_states, planets_df, buildings_df


empire_planet_states, empire_planets_df, empire_planet_buildings_df = build_padded_empire_planets(
    live_planet_states,
    max_planets=MAX_EMPIRE_PLANETS,
    empty_planet_size=EMPTY_COLONY_SIZE_FIELDS,
    empty_temp_c=EMPTY_COLONY_DEFAULT_TEMP_C,
)

planet_sizes_arr: np.ndarray = empire_planets_df["fields_max"].to_numpy(dtype=np.int64)
planet_fields_used_arr: np.ndarray = empire_planets_df["fields_used"].to_numpy(dtype=np.int64)
planet_fields_remaining_arr: np.ndarray = empire_planets_df["fields_remaining"].to_numpy(dtype=np.int64)

_colonized_mask = ~empire_planets_df["is_empty_slot"]
EMPIRE_PLANET_STATE_SUMMARY: dict[str, Any] = {
    "max_empire_planets": int(MAX_EMPIRE_PLANETS),
    "empty_colony_size_fields": int(EMPTY_COLONY_SIZE_FIELDS),
    "colonized_planets": int(_colonized_mask.sum()),
    "empty_colony_slots": int((~_colonized_mask).sum()),
    "total_empire_slots": int(len(empire_planets_df)),
    "total_fields_capacity": int(np.sum(planet_sizes_arr)),
    "total_fields_used": int(np.sum(planet_fields_used_arr)),
    "total_fields_remaining": int(np.sum(planet_fields_remaining_arr)),
    "mean_planet_size_fields": float(np.mean(planet_sizes_arr)) if planet_sizes_arr.size > 0 else 0.0,
}

GAME_PARAMS_TABLES["empire_planets"] = empire_planets_df
GAME_PARAMS_TABLES["empire_planet_buildings"] = empire_planet_buildings_df

print(
    f"Empire Planet Roster: {EMPIRE_PLANET_STATE_SUMMARY['colonized_planets']} colonized + "
    f"{EMPIRE_PLANET_STATE_SUMMARY['empty_colony_slots']} empty ({EMPTY_COLONY_SIZE_FIELDS}-field) slots = "
    f"{EMPIRE_PLANET_STATE_SUMMARY['total_empire_slots']} total | "
    f"Fields: {EMPIRE_PLANET_STATE_SUMMARY['total_fields_used']:,} used / "
    f"{EMPIRE_PLANET_STATE_SUMMARY['total_fields_capacity']:,} total "
    f"({EMPIRE_PLANET_STATE_SUMMARY['total_fields_remaining']:,} free)"
)
print("\n[empire_planets_df — Full Empire Planet Sizes & States]")
print(
    empire_planets_df[
        [
            "slot_index",
            "planet_id",
            "status",
            "name",
            "coords",
            "temperature_c",
            "fields_used",
            "fields_max",
            "fields_remaining",
            "resources_spice_eq",
            "energy_net",
            "total_building_levels",
        ]
    ].to_string(index=False)
)
print("\n[empire_planet_buildings_df — Active Colony & Empty Slot Building Matrix (b1..b14)]")
print(
    empire_planet_buildings_df[
        ["slot_index", "coords", "status", "fields_used", "fields_max"]
        + [f"b{i}" for i in range(1, 15)]
    ].to_string(index=False)
)

## 9. Print Every Generated Data Set (One Cell per Section)

Each cell below prints, in full, everything that the matching section above generated: every `pandas` DataFrame (all rows and all columns), every `numpy` array (all elements), and every parameter dictionary and scalar. Nothing is shortened with `...`, large numbers are written out instead of being switched to scientific notation, and every printout starts with the date and time at which it was printed.

- Sections 0, 0.1 and 1 only install packages, import modules and define helpers, so they have no data to print.
- The static `SQL_*` query strings and `*_DF_COLUMNS` column-name tuples are source constants, not generated data, and are not repeated here.
- A plain-Python list that holds the same rows or values as a table or array printed in the same cell (for example `attack_records` and `attacks_df`) is listed with its size and a note instead of being printed twice.
- All printed text passes through `redact_sensitive`, so a session token can never appear in a printout.

### 9.1 Section 2 Data — MER SQLite Telemetry

`telemetry_source`, `origin_planet_coords`, `fleet_slot_capacity`, the `telemetry_bundle` scalars and colony levels, `attacks_df`, `recycles_df`, and `attack_dispatch_ts_arr`.

In [ ]:
print_section_data(
    "Section 2 data: MER-aligned SQLite telemetry",
    [
        ("telemetry_source", telemetry_source),
        ("origin_planet_coords", origin_planet_coords),
        ("fleet_slot_capacity", fleet_slot_capacity),
        (
            "telemetry_bundle (every key except the three row lists, which are the tables below)",
            {
                k: v
                for k, v in telemetry_bundle.items()
                if k not in ("records", "recycle_rows", "dispatch_timestamps")
            },
        ),
        ("attacks_df", attacks_df),
        ("attack_records", attack_records, "same rows as attacks_df"),
        ("recycles_df", recycles_df),
        ("recycle_records", recycle_records, "same rows as recycles_df"),
        ("attack_dispatch_ts_arr", attack_dispatch_ts_arr),
        ("attack_dispatch_timestamps", attack_dispatch_timestamps, "same values as attack_dispatch_ts_arr"),
    ],
)

### 9.2 Section 3 Data — Attack Resource Means & Standard Deviations

`METRIC_KEYS`, `attack_resource_stats_df`, the five per-resource mean / standard-deviation dictionaries, the recycle harvest mean and standard deviation, and `winning_attacks_df`.

In [ ]:
print_section_data(
    "Section 3 data: attack resource means and standard deviations",
    [
        ("METRIC_KEYS", METRIC_KEYS),
        ("attack_resource_stats_df", attack_resource_stats_df),
        ("attack_resource_mean", attack_resource_mean),
        ("attack_resource_std", attack_resource_std),
        ("attack_resource_pop_std", attack_resource_pop_std),
        ("winning_attack_resource_mean", winning_attack_resource_mean),
        ("winning_attack_resource_std", winning_attack_resource_std),
        ("recycle_spice_eq_mean", recycle_spice_eq_mean),
        ("recycle_spice_eq_std", recycle_spice_eq_std),
        ("winning_attacks_df", winning_attacks_df),
    ],
)

### 9.3 Section 4 Data — Attack Flight & Cadence Times

`attack_time_stats_df`, the nine exact mean / standard-deviation scalars, and the `one_way_times_arr`, `round_trip_times_arr`, and `inter_attack_intervals_arr` arrays.

In [ ]:
print_section_data(
    "Section 4 data: attack flight and cadence times",
    [
        ("attack_time_stats_df", attack_time_stats_df),
        (
            "exact time statistics in seconds",
            {
                "attack_one_way_time_mean_sec": attack_one_way_time_mean_sec,
                "attack_one_way_time_std_sec": attack_one_way_time_std_sec,
                "attack_one_way_time_pop_std_sec": attack_one_way_time_pop_std_sec,
                "attack_time_mean_sec": attack_time_mean_sec,
                "attack_time_std_sec": attack_time_std_sec,
                "attack_time_pop_std_sec": attack_time_pop_std_sec,
                "inter_attack_time_mean_sec": inter_attack_time_mean_sec,
                "inter_attack_time_std_sec": inter_attack_time_std_sec,
                "inter_attack_time_pop_std_sec": inter_attack_time_pop_std_sec,
            },
        ),
        ("one_way_times_arr", one_way_times_arr),
        ("one_way_times_sec", one_way_times_sec, "same values as one_way_times_arr"),
        ("round_trip_times_arr", round_trip_times_arr),
        ("round_trip_times_sec", round_trip_times_sec, "same values as round_trip_times_arr"),
        ("inter_attack_intervals_arr", inter_attack_intervals_arr),
        ("inter_attack_intervals_sec", inter_attack_intervals_sec, "same values as inter_attack_intervals_arr"),
    ],
)

### 9.4 Section 5 Data — Attack Success Rate & Simulation Parameters

`attack_summary_stats_df`, the success-rate and expected-yield scalars, `attack_class_stats_df` / `per_class_success_rate`, `attack_target_stats_df` / `per_target_success_rate`, `ATTACK_SIM_PARAMS`, and the `ATTACK_STATS_TABLES` registry.

In [ ]:
print_section_data(
    "Section 5 data: attack success rate and simulation parameters",
    [
        ("attack_summary_stats_df", attack_summary_stats_df),
        (
            "success-rate and expected-yield scalars",
            {
                "attack_attempts": attack_attempts,
                "attack_wins": attack_wins,
                "attack_losses": attack_losses,
                "attack_success_rate": attack_success_rate,
                "attack_success_rate_std_err": attack_success_rate_std_err,
                "attack_success_rate_smoothed": attack_success_rate_smoothed,
                "expected_raids_per_slot_hr": expected_raids_per_slot_hr,
                "expected_spice_eq_per_slot_hr": expected_spice_eq_per_slot_hr,
                "expected_net_spice_eq_per_slot_hr": expected_net_spice_eq_per_slot_hr,
                "max_empire_gross_spice_eq_hr": max_empire_gross_spice_eq_hr,
            },
        ),
        ("attack_class_stats_df", attack_class_stats_df),
        ("per_class_success_rate", per_class_success_rate),
        ("attack_target_stats_df", attack_target_stats_df),
        ("per_target_success_rate", per_target_success_rate),
        ("ATTACK_SIM_PARAMS", ATTACK_SIM_PARAMS),
        ("ATTACK_STATS_TABLES", ATTACK_STATS_TABLES),
    ],
)

### 9.5 Section 6 Data — Reference Context & Merchant Parity

`GAME_PARAM_CONTEXT`, the table level caps and build / research / shipyard time divisors, and `merchant_rates_df`.

In [ ]:
print_section_data(
    "Section 6 data: reference context and Merchant parity",
    [
        ("GAME_PARAM_CONTEXT", GAME_PARAM_CONTEXT),
        (
            "table level caps and time divisors",
            {
                "MAX_BUILDING_TABLE_LEVEL": MAX_BUILDING_TABLE_LEVEL,
                "MAX_RESEARCH_TABLE_LEVEL": MAX_RESEARCH_TABLE_LEVEL,
                "BUILDING_TIME_DIVISOR": BUILDING_TIME_DIVISOR,
                "RESEARCH_TIME_DIVISOR": RESEARCH_TIME_DIVISOR,
                "SHIPYARD_TIME_DIVISOR": SHIPYARD_TIME_DIVISOR,
            },
        ),
        ("merchant_rates_df", merchant_rates_df),
    ],
)

### 9.6 Section 6.1 Data — Buildings

`buildings_catalog_df` (one row per building) and `buildings_levels_df` (every level `L1`–`L50` of every building, all 28 columns).

**Precision of high-level costs.** `game_data` copies costs from the wiki, which rounds every cost of `1,000,000,000` or more. Checked against exact doubling for the `2.0x` buildings: every cost below `10^9` is exact, larger costs are up to 0.42% off, and from `10^15` only 5 significant digits remain (hence values such as `281470000000000032768`).

In [ ]:
print_section_data(
    "Section 6.1 data: buildings",
    [
        ("buildings_catalog_df", buildings_catalog_df),
        ("buildings_levels_df", buildings_levels_df),
    ],
)

### 9.7 Section 6.2 Data — Technologies

`technologies_catalog_df` (one row per research) and `technologies_levels_df` (every level `L1`–`L30` of every research).

The wiki rounding described in 9.6 applies here too: research costs of `1,000,000,000` or more are up to 0.39% off exact doubling, and costs below `10^9` are exact.

In [ ]:
print_section_data(
    "Section 6.2 data: technologies",
    [
        ("technologies_catalog_df", technologies_catalog_df),
        ("technologies_levels_df", technologies_levels_df),
    ],
)

### 9.8 Section 6.3 Data — Spaceships

`ships_df` (all 15 ships, all 34 columns).

In [ ]:
print_section_data(
    "Section 6.3 data: spaceships",
    [
        ("ships_df", ships_df),
    ],
)

### 9.9 Section 6.4 Data — Planetary Defenses & Parameter Registry

`defenses_df` (all 11 defenses), `game_params_summary_df` (the summary exactly as Section 6.4 built it, covering the 7 tables registered at that point), and the `GAME_PARAMS_TABLES` registry (name and shape of every table registered by the time this cell runs, including the ones added by Sections 7 and 8).

In [ ]:
print_section_data(
    "Section 6.4 data: planetary defenses and parameter registry",
    [
        ("defenses_df", defenses_df),
        ("game_params_summary_df", game_params_summary_df),
        ("GAME_PARAMS_TABLES", GAME_PARAMS_TABLES),
    ],
)

### 9.10 Section 7.1 Data — In-Memory Technology Requirement Graph

The contents of the in-memory `TECH_GRAPH` object (node and edge counts, `topological_order`, the raw `nodes` store, and the transitive `ancestors_of` / `descendants_of` closure of every node) together with `tech_graph_nodes_df` and `tech_graph_edges_df` (all 101 requirement edges).

In [ ]:
print_section_data(
    "Section 7.1 data: in-memory technology requirement graph",
    [
        (
            "TECH_GRAPH (TechnologyRequirementGraph summary)",
            {
                "nodes": len(TECH_GRAPH.nodes),
                "edges": sum(len(reqs) for reqs in TECH_GRAPH.prereqs_of.values()),
                "max_topological_layer": max(TECH_GRAPH.topological_layer.values()),
            },
        ),
        ("TECH_GRAPH.topological_order", list(TECH_GRAPH.topological_order)),
        ("TECH_GRAPH.nodes", pd.DataFrame(list(TECH_GRAPH.nodes.values()))),
        ("tech_graph_nodes_df", tech_graph_nodes_df),
        ("tech_graph_edges_df", tech_graph_edges_df),
        (
            "TECH_GRAPH.ancestors_of / TECH_GRAPH.descendants_of (transitive closure per node)",
            {
                node_key: {
                    "all_prerequisites": [
                        k for k in TECH_GRAPH.topological_order if k in TECH_GRAPH.ancestors_of[node_key]
                    ],
                    "all_unlocks": [
                        k for k in TECH_GRAPH.topological_order if k in TECH_GRAPH.descendants_of[node_key]
                    ],
                }
                for node_key in TECH_GRAPH.topological_order
            },
        ),
    ],
)

### 9.11 Section 7.2 Data — Unlock Path Plans

`unlock_targets_summary_df` (remaining unlock cost, time, and ordered prerequisite chain for all 62 entities), `phoenix_unlock_plan_df` (from the active colony), and `cold_phoenix_unlock_plan_df` (from an empty account).

In [ ]:
print_section_data(
    "Section 7.2 data: unlock path plans",
    [
        ("unlock_targets_summary_df", unlock_targets_summary_df),
        ("phoenix_unlock_plan_df", phoenix_unlock_plan_df),
        ("cold_phoenix_unlock_plan_df", cold_phoenix_unlock_plan_df),
    ],
)

### 9.12 Section 8.1 Data — Live Planet Sizes & States

The empire slot parameters (`MAX_EMPIRE_PLANETS`, `EMPTY_COLONY_SIZE_FIELDS`, `EMPTY_COLONY_DEFAULT_TEMP_C`), `live_planets_df`, and the full typed `live_planet_states` (one JSON line per owned planet with its resources, storage, energy, buildings, ships, defenses, and economy rates).

In [ ]:
print_section_data(
    "Section 8.1 data: live planet sizes and states",
    [
        (
            "empire slot parameters",
            {
                "MAX_EMPIRE_PLANETS": MAX_EMPIRE_PLANETS,
                "EMPTY_COLONY_SIZE_FIELDS": EMPTY_COLONY_SIZE_FIELDS,
                "EMPTY_COLONY_DEFAULT_TEMP_C": EMPTY_COLONY_DEFAULT_TEMP_C,
            },
        ),
        ("live_planets_df", live_planets_df),
        ("live_planet_states", live_planet_states),
    ],
)

### 9.13 Section 8.2 Data — Full Empire Roster with Empty 300-Field Slots

`EMPIRE_PLANET_STATE_SUMMARY`, `empire_planets_df`, `empire_planet_buildings_df` (all `b1`–`b21` levels per slot), the `planet_sizes_arr` / `planet_fields_used_arr` / `planet_fields_remaining_arr` arrays, and the full typed `empire_planet_states` (one JSON line per owned planet and per empty slot).

In [ ]:
print_section_data(
    "Section 8.2 data: full empire roster with empty 300-field slots",
    [
        ("EMPIRE_PLANET_STATE_SUMMARY", EMPIRE_PLANET_STATE_SUMMARY),
        ("empire_planets_df", empire_planets_df),
        ("empire_planet_buildings_df", empire_planet_buildings_df),
        ("planet_sizes_arr", planet_sizes_arr),
        ("planet_fields_used_arr", planet_fields_used_arr),
        ("planet_fields_remaining_arr", planet_fields_remaining_arr),
        ("empire_planet_states", empire_planet_states),
    ],
)